# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

## Lab | Agents and Tools
Building and evaluating a research assistant for a bank product team
This notebook implements the lab end to end. The assistant can use:
1. Calculator — safe arithmetic for numerical questions.
2. Internal-document retrieval — semantic search over the supplied bank documents.
3. Web search — Tavily search for current external information, with compact results.
It then measures tool selection and approximate token cost, tests conversation memory and recursion limits, and discusses a voice extension.

Important execution note: the calculator and document-grounding checks can run without paid services. Agent calls require an OpenAI API key; web search additionally requires a Tavily API key. The notebook detects missing credentials and skips only the dependent sections rather than failing halfway through.

# Lab | AI Agents, Web Research & Intelligent Tool Use

## Build a research agent that answers with evidence

This lab adapts the original *Agents and Tools* exercise. The core task is to build an agent that researches current and historical topics on the public web, returns structured Markdown, and makes its evidence auditable.

### Learning outcomes

By the end of the lab, you will be able to:

1. Configure a LangChain agent with web-search and calculator tools.
2. Write precise research prompts with scope, date boundaries, output structure, and source requirements.
3. Run three research exercises and render their answers as Markdown.
4. Inspect tool calls and token usage, and evaluate source quality.
5. Extend the agent to a practical scientific R&D use case.

### The three essential exercises

- **Exercise 1 — AI in Healthcare:** advancements in generative AI during calendar year 2025.
- **Exercise 2 — AI Startups Landscape:** emerging AI startups, products, and innovations during 2025.
- **Exercise 3 — Product Comparison:** OpenAI ChatGPT Team and Anthropic Claude Pro, including features, pricing, and reviews.

> Research-time rule: this lab may be run after 2025. The prompts therefore request a retrospective review of **1 January–31 December 2025**, require publication/event dates, and tell the agent not to present later developments as 2025 events. For product pricing and availability, the prompt requests the latest verifiable information at execution time and asks the agent to distinguish historical 2025 names from current product names.

### Setup
Setup and configuration
Install the dependencies in your environment before running the notebook

In [1]:
!pip install -U pandas python-dotenv openai langchain langchain-openai langchain-tavily langgraph sentence-transformers chromadb tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 242.6 kB/s  0:00:09m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 577.2/577.2 kB 265.0 kB/s  0:00:02eta 0:00:01
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.2.1
    Uninstalling python-dotenv-1.2.1:
      Successfully uninstalled python-dotenv-1.2.1
  Attempting uninstall: openai
    Found existing installation: openai 3.41.0
    Uninstalling openai-3.41.0:
      Successfully uninstalled openai-3.41.0
  Attempting uninstall: langchain-core━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [openai]
    Found existing installation: langchain-core 1.7.3━━━━━━━━━ 1/5 [openai]
    Uninstalling langchain-core-1.7.3:━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [openai]
      Successfully uninstalled langchain-core-1.7.3━━━━━━━━━━━ 1/5 [openai]
  Attempting uninstall: langgraph╺━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [langchain-core]
    Found existing installation: langgraph 1.2.18━━━━━━━━━━━━━ 2/5 [langchain-core]
    Uninstalling

The embedding model is downloaded the first time it is used. If that is not possible, the notebook falls back to a transparent TF-IDF retriever so that the retrieval workflow remains testable offline. The TF-IDF fallback is lexical rather than semantic; use the MiniLM + Chroma path for the intended lesson when dependencies and network access are available.

In [16]:
%pip install -U langchain-tavily langchain-classic

import os
from getpass import getpass

from dotenv import load_dotenv
from langchain_classic.agents import initialize_agent
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
import tiktoken

load_dotenv()
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass("Enter your Tavily API key: ").strip()
if not os.environ["TAVILY_API_KEY"]:
    raise ValueError("A valid TAVILY_API_KEY is required for web search.")
%pip install -U langchain-tavily

import warnings
warnings.filterwarnings("ignore")


tavily_search = TavilySearch(max_results=5)

llm = ChatOpenAI(model="gpt-4", temperature=0)
encoding = tiktoken.encoding_for_model("gpt-4")

def safe_search(query: str) -> str:
    result = tavily_search.run(query)
    result_text = str(result)

    tokens = encoding.encode(result_text)
    return encoding.decode(tokens[:3500])

tools = [
    Tool(
        name="TavilySafeSearch",
        func=safe_search,
        description="Search the web for current information.",
    )
]
from dotenv import load_dotenv
from langchain_core.tools import Tool
import tiktoken
from IPython.display import Markdown, display
load_dotenv()
tavily_search = TavilySearch(max_results=5)
from langchain_classic.agents import AgentType, initialize_agent
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
warnings.filterwarnings('ignore')

load_dotenv()
tavily_search = TavilySearch(max_results=5)
tavily_search = TavilySearch(max_results=5)

def safe_search(query: str) -> str:
    result = tavily_search.run(query)
    result_text = result.get("content", "") or str(result) if isinstance(result, dict) else str(result)
    tokens = encoding.encode(result_text)
    return encoding.decode(tokens[:3500])


tools = [
    Tool(
        name="TavilySafeSearch",
        func=safe_search,
        description="Search the web for current information.",
    )
]
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
)

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
conda create -n ai-research-agents python=3.11 -y
conda activate ai-research-agents
python -m pip install --upgrade pip
pip install -r ../requirements.txt

In [17]:
# Imports, environment variables, and reusable helpers

import os
import ast
import operator
import logging
import warnings
from pathlib import Path
from typing import Any

import tiktoken
from dotenv import load_dotenv
from IPython.display import display, Markdown
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_tavily import TavilySearch

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO)

# Load secrets from this notebook's directory.
NOTEBOOK_DIR = Path.cwd()
load_dotenv(NOTEBOOK_DIR / ".env")
# Also support running the notebook from the project root.
load_dotenv(NOTEBOOK_DIR.parent / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
MODEL = os.getenv("OPENAI_MODEL", "openai:gpt-5.4-nano")

print("OpenAI API key configured:", bool(OPENAI_API_KEY))
print("Tavily API key configured:", bool(TAVILY_API_KEY))
print("Model:", MODEL)

if not OPENAI_API_KEY or not TAVILY_API_KEY:
    print("\nLive agent execution is disabled until both API keys are configured.")

OpenAI API key configured: True
Tavily API key configured: True
Model: openai:gpt-5.4-nano


## Define the agent's tools
The web-search tool keeps only useful fields and limits the returned text to 1,200 tokens. This reduces repeated context sent to the model while preserving source titles, URLs, and evidence snippets.

The calculator uses a restricted Python abstract syntax tree (AST). It supports basic arithmetic without executing arbitrary Python code.

In [18]:
# Safe arithmetic calculator: no eval(), imports, attributes, or function calls.

_ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
    ast.Mod: operator.mod,
}

def _safe_calculate(node):
    if isinstance(node, ast.Expression):
        return _safe_calculate(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPERATORS:
        left = _safe_calculate(node.left)
        right = _safe_calculate(node.right)
        # Avoid unreasonable exponentiation.
        if isinstance(node.op, ast.Pow) and abs(right) > 10:
            raise ValueError("Exponent is too large.")
        return _ALLOWED_OPERATORS[type(node.op)](left, right)
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_safe_calculate(node.operand))
    raise ValueError("Only basic arithmetic is allowed.")

@tool
def calculator(expression: str) -> str:
    """Calculate a basic arithmetic expression using numbers and +, -, *, /, %, and **."""
    try:
        parsed = ast.parse(expression, mode="eval")
        result = _safe_calculate(parsed)
        return str(result)
    except Exception as exc:
        return f"Calculation refused: {exc}"

# Test the calculator directly.
print(calculator.invoke("182000 * 0.14"))
print(calculator.invoke("__import__('os').getcwd()"))

25480.000000000004
Calculation refused: Only basic arithmetic is allowed.


In [19]:
# Tavily search returns public web results. Keep source metadata and trim the payload.

if TAVILY_API_KEY:
    tavily_raw = TavilySearch(max_results=5)
else:
    tavily_raw = None

try:
    tokenizer = tiktoken.get_encoding("o200k_base")
except Exception:
    tokenizer = tiktoken.get_encoding("cl100k_base")

MAX_SEARCH_TOKENS = 1200

def _trim_to_tokens(text: str, limit: int = MAX_SEARCH_TOKENS) -> str:
    token_ids = tokenizer.encode(text)
    if len(token_ids) <= limit:
        return text
    return tokenizer.decode(token_ids[:limit]) + "\n[Search context truncated to control token usage.]"

@tool
def web_search(query: str) -> str:
    """Search the public web for recent or historical information. Use this for current facts, research updates, company information, product details, pricing, and independent reviews. Include source titles, URLs, and evidence snippets in your answer."""
    if tavily_raw is None:
        return "Web search is unavailable: configure TAVILY_API_KEY in your .env file."
    try:
        raw = tavily_raw.invoke({"query": query})
        # Tavily may return a dictionary containing a results list.
        results = raw.get("results", []) if isinstance(raw, dict) else raw
        compact = []
        for item in results:
            if not isinstance(item, dict):
                continue
            compact.append({
                "title": item.get("title", "Untitled"),
                "url": item.get("url", ""),
                "content": item.get("content", "")
            })
        return _trim_to_tokens(str(compact))
    except Exception as exc:
        return f"Web search failed: {type(exc).__name__}: {exc}"

# Direct tool smoke test. This performs one web search and prints its token count.
if tavily_raw is not None:
    sample_search = web_search.invoke("generative AI healthcare clinical research 2025")
    print("Trimmed search-result tokens:", len(tokenizer.encode(sample_search)))
    print(sample_search[:1500])
else:
    print("Skipping live Tavily smoke test because TAVILY_API_KEY is not configured.")

Trimmed search-result tokens: 1210
[{'title': '4.1K views · 17 reactions | Generative AI is set to revolutionize health care strategy and growth in 2025. With 59% of U.S. health care leaders already partnering with vendors for AI solutions, the impact will be profound.\n\nA snapshot of how Generative AI will enhance care in the year ahead: \n✔️ Clinical productivity \n✔️ Patient engagement \n✔️ Administrative efficiency \n✔️ Quality of care \n✔️ Technology and infrastructure \n✔️ Research and education\n\nLearn more about how AI is revolutionizing health care and discover other health care trends for the year ahead.\nhttps://bit.ly/40rdoTg | Cigna Healthcare', 'url': 'https://www.facebook.com/CIGNA/videos/generative-ai-is-set-to-revolutionize-health-care-strategy-and-growth-in-2025-wi/586082124282330', 'content': 'other health care trends for the year ahead. [...] now by searching “The Visibility Gap” wherever you stream your podcasts. #mentalhealth #burnout #mensmentalhealth [...] Gen

## Create the research agent
The system instructions tell the agent when to use each tool and how to handle evidence. The agent must not invent sources, dates, prices, or review scores. If it cannot verify a claim, it must say so.

The agent is instructed to return Markdown with clear headings, tables where useful, and source links adjacent to claims.

In [20]:
SYSTEM_PROMPT = """
You are a careful research analyst. Answer the user's research question using the available tools.

TOOL SELECTION
- Use web_search for current information, historical developments, company/product facts, pricing, reviews, and any claim that needs external evidence.
- Use calculator for arithmetic, percentages, totals, and transparent numeric comparisons.
- Do not claim to have searched the web if web_search was not called.

EVIDENCE AND ACCURACY
- Cite factual claims with numbered source references such as [1] and provide the corresponding full URL in a Sources section.
- Prefer primary sources (official announcements, regulators, company documentation, research papers) and reputable independent reporting.
- Distinguish the date an event occurred from the date an article was published.
- Never invent a company, product, price, review, metric, date, or URL.
- Flag uncertainty, conflicting sources, missing evidence, and limitations explicitly.
- For time-sensitive facts, state the date on which the information was checked.
- Do not treat search snippets as conclusive evidence when the underlying source is unclear.

OUTPUT
- Use clean, well-structured Markdown.
- Start with a concise executive summary.
- Use tables for comparisons or lists with repeated attributes.
- Include a concise conclusion and a Sources section.
- Be specific, factual, and avoid unsupported superlatives.
"""

if OPENAI_API_KEY and TAVILY_API_KEY:
    research_agent = create_agent(
        model=MODEL,
        tools=[web_search, calculator],
        system_prompt=SYSTEM_PROMPT
    )
    print("Research agent initialized.")
else:
    research_agent = None
    print("Agent not initialized. Add both API keys, then rerun this cell.")

Research agent initialized.


## Reusable execution and reporting function
The helper below extracts the final answer, tool calls, model-call count, and token usage when the model provider returns usage metadata. Token totals are observational; monetary cost is not hard-coded because model prices can change and depend on the account/model.

In [21]:
def run_research(agent, prompt: str) -> dict[str, Any]:
    """Run one research prompt and return its answer plus execution diagnostics."""
    if agent is None:
        raise RuntimeError("Agent is not initialized. Configure OPENAI_API_KEY and TAVILY_API_KEY.")

    result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
    messages = result.get("messages", [])
    ai_messages = [m for m in messages if type(m).__name__ == "AIMessage"]

    tool_calls = [
        call.get("name")
        for message in ai_messages
        for call in (getattr(message, "tool_calls", None) or [])
    ]

    input_tokens = 0
    output_tokens = 0
    for message in ai_messages:
        usage = getattr(message, "usage_metadata", None) or {}
        input_tokens += usage.get("input_tokens", 0) or 0
        output_tokens += usage.get("output_tokens", 0) or 0

    final_message = messages[-1] if messages else None
    answer = getattr(final_message, "content", str(final_message))
    if isinstance(answer, list):
        answer = "\n".join(
            part.get("text", str(part)) if isinstance(part, dict) else str(part)
            for part in answer
        )

    return {
        "answer": answer,
        "tools": tool_calls,
        "model_calls": len(ai_messages),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens
    }

def display_run(label: str, result: dict[str, Any]) -> None:
    print(f"## {label}")
    print("Tools called:", result["tools"])
    print("Model calls:", result["model_calls"])
    print("Tokens (input/output/total):",
          result["input_tokens"], result["output_tokens"], result["total_tokens"])
    display(Markdown(result["answer"]))

### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

## Exercise 1: AI in Healthcare
Goal: Investigate and summarize the most important advancements in generative AI applied to healthcare during calendar year 2025.

The prompt explicitly defines the time window, asks for evidence and impact, and guards against mixing 2026 developments into the 2025 retrospective.

In [51]:
# Your prompt here
prompt_1 = """
Research the most important advancements in generative AI applied to healthcare
that occurred or were announced between 1 January and 31 December 2025.

Research requirements:
1. Search for recent, verifiable developments across clinical documentation,
   ambient clinical assistants, medical imaging, drug discovery, clinical trials,
   patient communication, medical education, and healthcare operations.
2. Identify 6–10 significant developments. For each, report:
   - organization(s) and system/product;
   - what was technically or clinically new;
   - healthcare setting and intended users;
   - evidence of impact (study results, validation, deployment scale, or regulatory status);
   - event date and source publication date.
3. Distinguish peer-reviewed clinical evidence from company announcements,
   pilots, demonstrations, and unvalidated claims.
4. Include relevant regulatory or safety developments from 2025, where supported.
5. Explain major limitations: hallucinations, bias, privacy, security,
   workflow integration, accountability, and evaluation.
6. Conclude with 3 evidence-based trends and 3 unresolved challenges.

Time boundary: include developments from calendar year 2025 only.
Do not present a 2026 development as a 2025 achievement. If a 2025 claim
cannot be verified, omit it or label the uncertainty.

Output format: clean Markdown with:
- Executive summary
- A table of 6–10 developments
- Evidence and safety considerations
- Key trends and unresolved challenges
- Conclusion
- Sources (numbered, with direct URLs)

Use authoritative sources first: peer-reviewed journals, regulators,
health systems, universities, and official company announcements.
State the date on which this research was performed.
"""

# Run Exercise 1.
if research_agent:
    response_1 = run_research(research_agent, prompt_1)
    display_run("Exercise 1 — AI in Healthcare", response_1)
else:
    print("Configure API keys and initialize the agent to run Exercise 1.")

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


## Exercise 1 — AI in Healthcare
Tools called: ['web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search']
Model calls: 8
Tokens (input/output/total): 24445 2349 26794


**Date of research:** 2 Nov 2026.  
## Executive summary  
Across calendar year 2025, the clearest “generative AI in healthcare” advancements with verifiable impact were (1) stronger clinical-evidence for **ambient AI scribes** reducing documentation burden/burnout, (2) continued **deployment expansion** of ambient documentation in real health-system workflows (mostly company/vendor-reported, with studies increasingly validating outcomes), (3) major **regulatory maturation** for AI-enabled medical devices (notably FDA lifecycle/change-control guidance), and (4) continued acceleration of **AI in imaging and drug discovery**, though many 2025 imaging/drug “generative” claims remain pilot/demonstration-grade without peer-reviewed clinical outcome validation. [https://pmc.ncbi.nlm.nih.gov/articles/PMC12492056] [https://www.fda.gov/media/166704/download]  

## Table: 6–10 significant 2025 developments (generative AI in healthcare)
| # | Organization(s) & system/product | What was technically/clinically new | Setting & intended users | Evidence of impact & regulatory status | Event date (2025) & source pub date |
|---:|---|---|---|---|---|
| 1 | **Schwamm et al. / research team** (ambient AI scribes evaluated in study; product not uniquely determinable from the source) | Randomized evaluation showing ambient AI scribe use associated with **lower documentation-burden-related burnout** | Clinicians using EHR documentation workflows (multispecialty) | **Peer-reviewed (JAMA Netw Open)**: burnout decreased significantly (51.9%→38.8%), plus improvements on multiple clinician experience scales | **Event/analysis period** reported as within study timeline; **pub date 2025-10-02** [https://pmc.ncbi.nlm.nih.gov/articles/PMC12492056] |
| 2 | **Olson et al. (JAMA Netw Open)** on ambient AI scribes | Demonstrated clinician-reported outcome improvements tied to ambient scribe documentation (reduced burnout; improved perceived ability to focus) | Health-system clinicians generating notes from ambient audio | **Peer-reviewed** with quantitative clinician outcomes | **Pub date 2025-10-02** [https://pmc.ncbi.nlm.nih.gov/articles/PMC12492056] |
| 3 | **Abridge** (company product) | Continued scaling/validation efforts for ambient clinical documentation (“ambient AI scribes” category) | Health systems deploying ambient documentation for clinicians | **Third-party peer-reviewed evidence available for the class of ambient scribes**, while this specific row is **company announcement context** (not a stand-alone clinical-outcome study) | Use **company-site discovery only** (exact 2025 dates not confirmed from the provided sources); included here only as “announcement context,” not as clinical proof [https://www.abridge.com] |
| 4 | **FDA** | Final guidance for **Predetermined Change Control Plans (PCCP)** for AI-enabled device software functions—formalizing lifecycle/allowed updates without repeated submissions | Applicable to manufacturers of AI-enabled SaMD / AI-DSFs | **Regulatory artifact (authoritative)**: final guidance issued **Aug 18, 2025** | **Issued 2025-08-18**; publication in FDA guidance document [https://www.fda.gov/media/166704/download] |
| 5 | **FDA / regulated community** | 2025-era surge in **AI/ML medical device clearances** indicates operational momentum for AI deployment; while not “generative” exclusively, it supports ecosystem readiness for generative SaMD | Broad clinical device ecosystem (diagnostics, imaging, etc.) | **Industry dataset summary** (not peer-reviewed clinical outcomes; evidence is regulatory-clearance counts) | **Published summary covering 2025** [https://innolitics.com/articles/year-in-review-ai-ml-medical-device-k-clearances] |
| 6 | **FDA AI-Enabled Medical Device List** (census-based analysis) | Regulatory tracking showing cumulative authorizations through 2025 and identifying 2025 as busiest year (again, not exclusively generative) | SaMD across specialties | **Census/analysis** based on FDA list (method not peer-reviewed in the excerpt provided) | **Analysis cites “final decisions through 31 Dec 2025”**; source accessed externally [https://aimocs.com/statistics/fda-cleared-ai-devices-by-year-and-specialty] |
| 7 | Imaging AI vendors ecosystem (context) | Continued focus on **generative modeling** (e.g., diffusion/GANs) for reconstruction/augmentation; emphasizes need for validation to avoid “hallucinations” in imaging | Radiology workflows; intended users radiologists/technologists (intended use varies by product) | **Regulatory and review-level context**, not specific 2025 cleared generative imaging product outcomes from peer-reviewed trials in provided sources | General 2025 trends article (not a single dated clinical event) [https://intuitionlabs.ai/articles/ai-radiology-trends-2025] |
| 8 | Drug discovery/clinical trials space (context gap) | Many 2025 “generative AI for drug discovery/trials” announcements exist, but **specific 2025 peer-reviewed clinical impact items were not verifiably retrieved** in the provided searches | Biopharma discovery & clinical trial operations | **Omitted from “verified 2025 achievements”** due to verification gap per your constraint | Not included as a verified item |

> **Note:** I verified strong clinical evidence for ambient AI scribes via peer-reviewed literature; for other categories (imaging, drug discovery, trials, patient communication, education, operations), the provided external sources were primarily trend/regulatory context rather than a set of fully verifiable, dated, peer-reviewed 2025 developments. If you want, I can run additional targeted searches per category (e.g., “2025 diffusion reconstruction FDA cleared randomized trial”, “2025 generative AI drug discovery clinical candidate trial results”, etc.) to fill rows 8–10 with fully supported items.

## Evidence and safety considerations
- **Peer-reviewed clinical evidence (ambient AI scribes):** JAMA Network Open reports significant reductions in burnout and improvements in clinician experience measures associated with ambient scribe use. [https://pmc.ncbi.nlm.nih.gov/articles/PMC12492056]  
- **Regulatory/safety (lifecycle & changes):** FDA’s Aug 18, 2025 PCCP final guidance addresses how manufacturers can pre-specify intended AI changes across the product lifecycle, which is directly relevant to safety/quality control after deployment. [https://www.fda.gov/media/166704/download]  
- **Major limitations (cross-cutting):** even when clinicians report workflow benefits, generative systems can still produce **hallucinations**, exhibit **bias**, and create **privacy/security/accountability** risks; robust evaluation and postmarket monitoring remain necessary (these risks are not quantified in the ambient scribe paper excerpt, but are standard considerations for generative clinical AI). [https://pmc.ncbi.nlm.nih.gov/articles/PMC12492056]  

## Key trends and unresolved challenges
**Trends (evidence-based from retrieved sources):**  
1) **Clinical documentation burden relief** becomes measurable with controlled/peer-reviewed outcomes for ambient scribes. [https://pmc.ncbi.nlm.nih.gov/articles/PMC12492056]  
2) **Regulatory lifecycle management** (change control / monitoring) becomes more explicit, enabling safer iteration of AI systems after clearance. [https://www.fda.gov/media/166704/download]  
3) **Scale-up of AI-enabled devices** continues rapidly (regulatory clearances increasing in 2025), supporting broader adoption readiness. [https://innolitics.com/articles/year-in-review-ai-ml-medical-device-k-clearances]  

**Unresolved challenges:**  
1) **Hallucination and clinical error risk** in open-ended generation (especially outside documentation, e.g., imaging interpretation or drug reasoning). [https://intuitionlabs.ai/articles/ai-radiology-trends-2025]  
2) **Bias and equity evaluation** across demographic subgroups (requirement implied by regulatory lifecycle focus; not demonstrated in the sources retrieved here). [https://www.fda.gov/media/166704/download]  
3) **Privacy/security and accountability** in real EHR/audio/video pipelines (not quantified in the retrieved peer-reviewed sources). [https://pmc.ncbi.nlm.nih.gov/articles/PMC12492056]  

## Conclusion  
The most verifiable 2025 advancements in generative AI for healthcare (from the sources retrieved here) are concentrated in **ambient clinical documentation**, where peer-reviewed results show reduced burnout and improved clinician experience, alongside **FDA’s 2025 maturation of AI lifecycle/change-control guidance** that supports safer deployment and updates. For imaging, drug discovery, trials, and patient-facing use cases, I found credible 2025 trend/regulatory context, but not enough **fully verifiable, dated, peer-reviewed clinical-impact developments** in the provided searches to responsibly list as “most important advancements” without risking unverifiable claims.  

## Sources  
1. https://pmc.ncbi.nlm.nih.gov/articles/PMC12492056  
2. https://www.fda.gov/media/166704/download  
3. https://innolitics.com/articles/year-in-review-ai-ml-medical-device-k-clearances  
4. https://aimocs.com/statistics/fda-cleared-ai-devices-by-year-and-specialty  
5. https://intuitionlabs.ai/articles/ai-radiology-trends-2025  
6. https://www.abridge.com

In [24]:
# Run it
if research_agent is None:
    raise RuntimeError("Research agent is not initialized. Configure the API keys and rerun its setup cell.")

response_1 = run_research(research_agent, prompt_1)["answer"]

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [25]:
display(Markdown(response_1))


## Executive summary (checked on **2026-11-02**)
Between **1 Jan–31 Dec 2025**, generative AI in healthcare saw important progress across (1) **clinical documentation/ambient scribing** with emerging real-world and study evidence, (2) **drug discovery** moving from lab to **randomized phase 2 proof-of-concept** for an AI-discovered therapy, (3) **patient communication workflows** using generative AI–drafted responses, (4) **medical education** with LLM-based virtual patient/simulation systems, and (5) **regulatory/safety infrastructure** evolving for AI-enabled medical devices (including lifecycle change control).  

Notably, **peer-reviewed clinical evidence** was more robust in documentation burden/burnout and patient communication than in imaging or large-scale operations, where many items are still pilots/demos without strong published outcomes.

---

## Key 2025 advancements (6–10 developments)

> **Evidence types used below:**  
> **Peer-reviewed clinical evidence** = published in peer-reviewed journals and reports study results.  
> **Regulatory status** = FDA guidance/clearance/authorization or regulator publication.  
> **Company announcement/pilot** = less rigorous unless accompanied by peer-reviewed evidence.

| # | Development (orgs + system/product) | What was technically/clinically new | Healthcare setting & intended users | Evidence of impact (2025) | Event date | Source publication date |
|---:|---|---|---|---|---|---|
| 1 | **Microsoft (Dragon Copilot / DAX ambient listening)** | Unified **ambient clinical capture** + fine-tuned generative AI for drafting clinical documentation, with clinician review before charting | Ambulatory clinical settings; **clinicians** documenting in the EHR | **Peer-reviewed multicenter QI study:** burnout decreased **51.9% → 38.8%** after 30 days; improvements in cognitive task load and after-hours documentation time | Trial/rollout period leading to post-intervention results in 2025 (study reports 30-day outcome; exact start not fully specified in abstract) | JAMA Netw Open article dated **2025-10-02** (published online/issues) [1] |
| 2 | **Olson et al. (study of ambient AI scribes; evaluated ambient scribe platform)** | Demonstrated measurable clinician well-being and workflow-related outcomes from ambient scribing deployment | Ambulatory care; **263 ambulatory clinicians** across 6 healthcare systems | **Peer-reviewed:** odds ratio for burnout **0.26 (95% CI 0.13–0.54)**; improvements on several secondary outcomes | Intervention period: **30 days** (within study design) | **2025-10-02** [1] |
| 3 | **Insilico Medicine (generative AI-discovered drug): Rentosertib / ISM001-055 (TNIK inhibitor)** | Generative AI used in drug discovery/design; first strong clinical “discovered with genAI” proof-of-concept in humans | Idiopathic pulmonary fibrosis; **patients in multicenter randomized phase 2a** | **Peer-reviewed Nature Medicine:** randomized phase 2a trial reporting safety/tolerability and efficacy signals; placebo-controlled design | **2025-06-03** (published online; trial completed earlier but publication is the verifiable anchor) | **2025-06-03** [2] |
| 4 | **Patient communication: Mandal et al. (generative AI–drafted responses in patient-provider communication)** | Adoption/utilization study of generative AI–drafted responses used in patient messaging workflows | Patient-provider messaging channels; **healthcare organizations + clinicians** managing inbox/messages | **Peer-reviewed (npj Digital Medicine):** evaluates utilization patterns and considerations (not a single “clinical outcome trial,” but direct workflow/implementation evidence) | Received **11 Apr 2025**, accepted **23 Aug 2025** (study time not fully specified) | **2025-10-02** [3] |
| 5 | **Medical education: LLM-based digital patient systems (npj Digital Medicine)** — “A large language model digital patient system enhances ophthalmology history taking skills” | LLM “digital patient” simulation for history-taking practice | Ophthalmology education; **medical learners/students** training history-taking | **Peer-reviewed:** published evidence that LLM-driven virtual patients improved history-taking skills (quantitative outcomes reported in paper) | Acceptance **24 Jun 2025**; published after | **2025-08-04** [4] |
| 6 | **Medical education validation pilot: ECOSBot (generative AI tool; GPT-4o) for nephrology OSCE simulation** | Generative AI tool designed to simulate standardized patients/examiners for OSCEs | Nephrology-focused **OSCE training/assessment**; **students/residents/examiners** in training contexts | **Pilot validation/technical validation** published in 2025 (not full clinical efficacy outcomes like an RCT) | Development/validation conducted before publication | **2025** (article in Clin Kidney J volume/issue; see publication record) [5] |
| 7 | **Regulatory/safety development: FDA predetermined change control plans (PCCPs) for AI-enabled device software functions** | FDA finalized guidance enabling certain planned AI software modifications via a pre-authorized change control framework (aiming lifecycle oversight for evolving AI) | Applies to **AI-enabled medical devices** (manufacturers and FDA review processes) | **Regulatory guidance (FDA):** provides marketing submission recommendations; important “safety governance” advancement | **Issued 2025-08-18** | FDA guidance document dated **Aug 18, 2025** [6] |
| 8 | **Regulatory foundation: FDA list updates for AI-enabled medical devices** (context for what’s being cleared/authorized in imaging and other specialties) | Ongoing publication of AI-enabled device authorizations/clearances informs landscape for generative AI-adjacent imaging/clinical tools | Intended for clinicians/patients via approved medical devices; broader stakeholder transparency | **Regulatory record:** FDA maintains an up-to-date public list with decision dates | Throughout 2025 | FDA list entry pages updated with 2025 decision dates (example entries visible on list) [7] |

### Notes on omitted/uncertain items
- I **did not include** additional 2025 claims about ambient scribing, imaging foundation models, or drug discovery unless I could anchor them to **peer-reviewed results, FDA/regulator documents, or clearly dated official announcements**.  
- Some categories you requested (e.g., **clinical trials automation**, **medical imaging generative models with 2025 peer-reviewed clinical validation**, **healthcare operations scheduling-scale deployments**) surfaced mostly as pilots/demos or non-verifiable claims in the search results I reviewed, so they were excluded to avoid unverified 2025 attribution.

---

## Evidence and safety considerations (what the 2025 evidence supports—and what it doesn’t)

### Hallucinations & clinical correctness
- Even when documentation productivity improves, **LLMs can still generate incorrect content**. Ambient scribe deployments typically rely on **human clinician review** before final record entry (critical for patient safety). The JAMA Network Open evidence shows outcome changes related to workload/burnout, but it does **not** prove perfect clinical accuracy of every drafted element [1].
  
### Bias & representativeness
- Patient communication and education tools can encode biases present in training data. The 2025 patient messaging utilization paper focuses on **use/utilization**; it does not by itself guarantee absence of bias in content generated for different patient groups [3].
  
### Privacy & data governance
- Ambient listening and EHR integration increase exposure to PHI. The FDA PCCP guidance (2025) is about **device software lifecycle oversight**, which indirectly supports safer governance practices, but it doesn’t eliminate privacy risks [6].

### Security (cybersecurity) & model update risks
- Model updates can change behavior over time. The PCCP framework is intended to help manufacturers manage predictable modifications with FDA review implications, reducing the chance that uncontrolled updates degrade safety/effectiveness [6].  
- However, cybersecurity specifics are handled in other regulatory guidance and QMS expectations; you would still need device-specific threat modeling, testing, and monitoring.

### Workflow integration & accountability
- Ambient scribing and message drafting can reduce burden only if integration into the **EHR and clinician inbox/workflows** is reliable and if there is clear accountability for errors (e.g., clinician sign-off vs automated charting). The evidence included here supports feasibility and some measured outcomes, but **organizational implementation quality** likely drives safety and effectiveness.

---

## Key trends (evidence-based from 2025 items above)

1. **Ambient/LLM documentation is moving from pilots to measurable clinician outcomes**  
   Peer-reviewed evidence in 2025 links ambient scribing to reduced burnout and related workflow measures in ambulatory settings [1].

2. **Generative AI is increasingly used for end-to-end drug discovery with human proof-of-concept**  
   Nature Medicine published a randomized phase 2a trial of an AI-discovered/design drug candidate (Rentosertib), representing a concrete translation milestone [2].

3. **Structured, operational use in patient communication and training is expanding**  
   Peer-reviewed utilization research on generative AI–drafted responses suggests growing incorporation into patient-provider messaging workflows [3], alongside LLM-based virtual patients improving learner competencies in medical education [4].

---

## Unresolved challenges (what remains hard)

1. **Reliable clinical accuracy and evaluation at the point of care**  
   Improved burnout/efficiency doesn’t automatically ensure that generated content is always correct, complete, and clinically appropriate in all edge cases.

2. **Bias, safety drift, and accountability under continuous model change**  
   Governance frameworks like FDA PCCPs help with lifecycle planning [6], but real-world safety drift, auditing, and accountability remain difficult—especially when systems are updated frequently.

3. **Privacy/security for systems that “listen” and integrate deeply with EHRs**  
   Ambient systems (and inbox copilots) create additional data flows. Robust privacy engineering, access control, secure architecture, and monitoring are still essential and can be challenging to implement uniformly.

---

## Conclusion
In 2025, the most defensible advancements in generative AI for healthcare were those with **peer-reviewed evidence** (notably ambient scribing outcomes [1,2] and education/communication utilization [3,4]) and those with **regulatory governance progress** (FDA PCCP guidance for evolving AI device software functions [6]). Many other promising areas you requested (especially imaging and operations at scale) require more **2025 peer-reviewed validation** or clearly documented regulatory/clinical deployment evidence to be included confidently.

---

## Sources
1. **Olson KD, et al.** “Use of Ambient AI Scribes to Reduce Administrative Burden and Professional Burnout.” *JAMA Network Open* (published/issue date **2025-10-02**). https://pmc.ncbi.nlm.nih.gov/articles/PMC12492056  
2. **Xu Z, et al.** “A generative AI-discovered TNIK inhibitor for idiopathic pulmonary fibrosis: a randomized phase 2a trial.” *Nature Medicine* (**2025-06-03**). https://pmc.ncbi.nlm.nih.gov/articles/PMC12353801  
3. **Mandal S, et al.** “Utilization of Generative AI-drafted Responses for Managing Patient-Provider Communication.” *npj Digital Medicine* (**2025-10-02**). https://www.nature.com/articles/s41746-025-01972-w  
4. **Luo MJ, et al.** “A large language model digital patient system enhances ophthalmology history taking skills.” *npj Digital Medicine* (**2025-08-04**). https://www.nature.com/articles/s41746-025-01841-6  
5. **Bentegeac R, et al.** “ECOSBot: a multicenter validation pilot study of a generative AI tool for OSCE-based nephrology training.” *Clinical Kidney Journal* (2025). https://academic.oup.com/ckj/article/18/10/sfaf308/8275750  
6. **FDA**. “Marketing Submission Recommendations for a Predetermined Change Control Plan for Artificial Intelligence-Enabled Device Software Functions.” Issued **Aug 18, 2025**. https://www.fda.gov/media/166704/download  
7. **FDA**. “List of Artificial Intelligence-Enabled Medical Devices” (public list; entries include 2025 decision dates). https://www.fda.gov/medical-devices/artificial-intelligence-enabled-medical-devices/list-artificial-intelligence-enabled-medical-devices  

--- 

If you want, I can do a second pass focused specifically on **medical imaging generative AI** and **clinical trial operations**, but I’ll only include items that I can substantiate with **2025 peer-reviewed clinical validation** or **official regulator/company documentation**.

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

## Exercise 2: AI Startups Landscape
Goal: Identify emerging AI startups associated with meaningful product innovation, funding, or commercial traction during 2025.

The prompt asks the agent to verify geography and funding rather than simply return a list of well-known AI companies.

In [26]:
# Your prompt here
prompt_2 = """
Build a researched overview of emerging AI startups and their innovations
during calendar year 2025 (1 January–31 December 2025).

Identify 8–12 startups across a useful mix of sectors, such as foundation
models, AI agents, healthcare, scientific discovery, robotics, developer
tools, enterprise AI, and climate/energy.

For every startup, provide:
- company name and headquarters country;
- founding year (only if verifiable);
- product/platform and the problem it solves;
- what is technically or commercially innovative;
- notable 2025 milestone (product launch, customer deployment, partnership,
  regulatory milestone, or funding round);
- funding amount, round, and date only when supported by a reliable source;
- primary source and at least one credible independent source where available.

Selection rules:
1. Prioritize genuinely emerging companies, not only established technology
   giants or mature public companies.
2. Do not describe a company as a 2025-founded startup unless its founding
   date is verified.
3. Clearly distinguish funding announced in 2025 from older funding.
4. Do not invent valuations, investors, customer numbers, or performance claims.
5. Explain why each company is worth watching, using evidence rather than hype.
6. Include a short section on geographic and sector patterns, plus limitations
   in the available public data.

Output format: clean Markdown with:
- Executive summary
- A comparison table (company, country, sector, product, 2025 milestone)
- 3–5 particularly interesting innovations explained in more detail
- Funding and market patterns
- Conclusion
- Sources (numbered, direct URLs)

Use sources published during 2025 where possible, and label the publication
date. State the date on which this research was performed.
"""

# Run Exercise 2.
if research_agent:
    response_2 = run_research(research_agent, prompt_2)
    display_run("Exercise 2 — AI Startups Landscape", response_2)
else:
    print("Configure API keys and initialize the agent to run Exercise 2.")

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


## Exercise 2 — AI Startups Landscape
Tools called: ['web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search']
Model calls: 4
Tokens (input/output/total): 77442 4047 81489


## Executive summary (research performed on **2026-11-02**)
This overview compiles **10 emerging (mostly private) AI startups** with notable **2025 milestones** across foundation models/agentic tooling, enterprise AI, healthcare, scientific discovery (energy/minerals), robotics, cybersecurity, and climate/energy. For each, I report what they build, why the approach is technically or commercially distinctive, and at least one **primary** and one **credible independent** source when available.

**Important limitations:** public 2025 data is uneven—many private startups disclose only some of the following: headquarters, exact founding year, customer deployments, funding amounts, or technical details. Where a datum (e.g., founding year or funding amount/date) could not be verified from reliable sources, I omit it rather than guess.

---

## Comparison table (8–12 startups)

| Company | HQ country | Sector | Product / platform (problem solved) | 2025 milestone |
|---|---|---|---|---|
| **Apptronik** | USA | Robotics (humanoids) | **Apollo** humanoid robots + fleet deployment platform for industrial/logistics labor automation | **Raised $350M Series A (Feb 13, 2025)**; expansion for Apollo deployments [1] [2] |
| **NEURA Robotics** | Germany | Robotics (cognitive/humanoid) | **Neuraverse** ecosystem + robots “see, hear and touch” for cognitive automation | **Raised €120M Series B (announced Jan 2025)** to accelerate cognitive robotics products [3] [4] |
| **X Square Robot** | China | Foundation models for embodied intelligence | **Wall-OSS** open-source embodied foundation model + supporting robot stack | **Released open-source WALL-OSS (Sept 2025)** (with associated Series A+ news) [5] [6] |
| **Tutor Intelligence** | USA | Robotics (warehouse automation) | **Robots-as-a-Service (RaaS)** for warehouse material-handling / logistics automation | **Closed $34M Series A (Dec 1, 2025)** to scale commercial robot fleet & central intelligence platform [7] [8] |
| **Skild AI** | USA | Robotics (robotics foundation model) | **Skild Brain** unified robotics foundation model intended to generalize across robot embodiments | **Unveiled “Skild Brain” omni-bodied model (July 2025)** (milestone emphasized by industry/registry sources) [9] |
| **Dropzone AI** | USA | AI agents (cybersecurity / SOC) | **AI SOC analysts** for alert triage & incident investigation augmentation | **$37M Series B (July 17, 2025)** to scale agentic SOC platform globally [10] |
| **Manus (Monica / Butterfly Effect)** | China | AI agents (developer/consumer agent platform) | Agentic tool to autonomously execute tasks across apps/workflows | **Launched Manus (March 6, 2025)** in invitation-only beta; broad media/testing coverage [11] [12] |
| **Hippocratic AI** | USA | Healthcare AI agents | Patient-facing **safe, non-diagnostic generative AI agents** for chronic care / outreach / medication checks | **Raised $126M Series C (announced Nov 3, 2025)** to expand clinically safe patient-facing agents [13] [14] |
| **UpDoc** | USA (implied) | Healthcare (clinical AI regulatory milestone) | Patient-facing clinical LLM for **insulin/medication management in type 2 diabetes** (SaMD pathway) | **FDA 510(k) clearance posted Dec 23, 2025 (K253281)** for device supporting patient-facing agentic workflow (public discussion/coverage) [15] [16] |
| **Eavor** | Canada (operates internationally) | Climate/energy (advanced geothermal) | **Eavor-Loop™ closed-loop geothermal** to deliver 24/7 clean heat & power | **Began supplying electricity to German grid (Dec 4, 2025)** from Geretsried facility [17] [18] |

> Note: “Sector” reflects the primary innovation theme for this snapshot. Some companies straddle multiple categories.

---

## 3–5 particularly interesting innovations (explained)

### 1) Open embodied foundation models for robotics: **X Square Robot (Wall-OSS)**
Instead of “robot-specific” training alone, X Square Robot emphasizes an **embodied foundation model** intended to generalize across tasks and robot platforms. The company’s **WALL-OSS** was positioned as open source, integrated into the **LeRobot** ecosystem, and supported by supporting repositories/docs—reducing barriers for downstream researchers and developers to adapt and deploy. [5] [6]

**Why watch:** open embodied model distribution could accelerate iteration cycles in robotics—similar to how open weights accelerated LLM research—while still targeting the harder “vision → action” end-to-end setting.

---

### 2) Enterprise-grade agent security operations: **Dropzone AI**
Dropzone AI’s approach is to operationalize AI agents in a high-stakes workflow: **SOC alert triage and investigation support**. Their 2025 funding/positioning aligns with the “AI SOC agent” trend toward productizing agents as operational infrastructure (not just chatbots). [10]

**Why watch:** if the company can maintain reliability, auditability, and integration into existing SOC tooling, it addresses a core obstacle to agent adoption: *operational safety and measured productivity gains*.

---

### 3) Safety-focused patient-facing health agents: **Hippocratic AI**
Hippocratic AI frames its platform around **clinically safe, patient-facing generative AI agents** for non-diagnostic tasks (e.g., outreach, wellness surveys, medication checks). A major 2025 milestone was a **$126M Series C** announcement tied to scaling and expansion. [13]

**Why watch:** healthcare adoption depends on risk controls, intended-use scoping, and integrations with provider workflows. A safety-first stance plus funding momentum can indicate a path to measurable outcomes and durability.

---

### 4) Regulatory pathway for patient-facing clinical LLMs: **UpDoc (K253281)**
A key “innovation” here is not just model capability but the **regulatory/compliance pathway**: **FDA 510(k) clearance posted Dec 23, 2025** for a device covering insulin/medication-management in type 2 diabetes. Public industry coverage treats it as a precedent-style signal that constrained, supervised patient-facing agentic systems can fit within FDA review frameworks. [15] [16]

**Why watch:** this could reduce uncertainty for other developers building patient-facing agent experiences—if the pathway proves repeatable.

---

### 5) 24/7 clean power from advanced geothermal: **Eavor (Geretsried grid supply)**
Eavor’s milestone is a shift from engineering validation toward **commercial power delivery**. Their Dec 4, 2025 announcement describes first electricity delivered to the German grid from the **Geretsried** facility. [17]

**Why watch:** advanced geothermal startups often struggle with the “last mile” of commissioning and grid reliability. A grid-connection milestone is a concrete risk-reduction event.

---

## Funding and market patterns (what 2025 data suggests)

### Funding patterns visible from the selected set
- **Robotics and agentic enterprise tools** show large 2025 rounds tied to scaling deployments:
  - Apptronik: **$350M Series A** to scale humanoid deployment plans [1] [2]
  - Dropzone AI: **$37M Series B** for global expansion and additional specialized agents [10]
  - Tutor Intelligence: **$34M Series A** to expand its robot fleet commercialization [7]
- **Healthcare** milestones combine **funding** (Hippocratic AI) with **regulatory evidence** (UpDoc clearance posted in 2025):
  - Hippocratic AI: **$126M Series C** [13]
  - UpDoc: **FDA 510(k) clearance posted Dec 23, 2025 (K253281)** [15] [16]
- **Climate/energy** milestones skew toward **project commissioning** rather than model-only advances:
  - Eavor’s grid-supply milestone is a commercial proof point [17] [18]

### Geographic + sector patterns (and why)
- **North America** dominates the samples for robotics commercialization, agentic enterprise tooling, and healthcare AI (HQ in USA for Apptronik, Dropzone AI, Tutor Intelligence, Hippocratic AI). This reflects the concentration of venture capital and commercialization infrastructure.
- **Europe** appears strongly via **German robotics (NEURA)** and **advanced geothermal commercialization in Germany (Eavor’s Geretsried)**—a reminder that “deployment milestones” can be as important as model performance.
- **Asia** is represented by **China** (Manus; X Square Robot), where rapid iteration on agent UX and embodied model releases is visible.

### Limitations in available public data
- **Founding years**: sometimes not clearly corroborated by primary sources in 2025-era reporting; I therefore omitted founding years when not reliably verifiable from what I reviewed.
- **Funding amounts/dates**: many startups disclose only partial figures or report strategic/“round” headlines; I only included amounts/dates when supported by reliable sources from the 2025 set.
- **Performance claims**: I avoided citing customer counts, ARR, safety metrics, or benchmarks unless a credible source provided them explicitly.

---

## Conclusion
Across 2025, emerging AI startups clustered into a few “real-world deployment” arcs:  
1) **embodied foundation models** (X Square Robot, plus broader robotics momentum),  
2) **agentic systems that plug into operational workflows** (Dropzone AI, Manus),  
3) **regulated and safety-scoped healthcare AI** (Hippocratic AI and UpDoc’s 510(k) clearance posted in 2025), and  
4) **climate/energy commercialization milestones** (Eavor and other energy-focused innovations).  

These themes matter because they move beyond demos toward **repeatable deployment**, **integration**, and—where necessary—**regulatory approval**.

---

## Sources
1. Apptronik press release: “Apptronik Raises $350 Million in Series A Funding” (Feb 13, 2025) — https://apptronik.com/news-collection/apptronik-raises-350-million-in-series-a-funding  
2. CNBC (independent) reporting on Apptronik Series A / Apollo context — https://www.cnbc.com/2026/02/11/apptronik-raises-520-million-at-5-billion-valuation-for-apollo-robot.html  
3. NEURA Robotics primary announcement (Series B, €120M) — https://neura-robotics.com/neura-robotics-secures-euro-120-million-series-b  
4. Bloomberg coverage of NEURA round (independent; subscriber may limit access) — https://www.bloomberg.com/news/articles/2025-01-15/humanoid-robot-maker-neura-robotics-raises-120-million-round  
5. Wikipedia overview of X Square Robot and WALL-OSS release context — https://en.wikipedia.org/wiki/X_Square_Robot  
6. Hugging Face documentation referencing WALL-OSS and LeRobot integration — https://huggingface.co/docs/lerobot/en/walloss  
7. Business Wire (via Yahoo Finance): Tutor Intelligence closes **$34M Series A** (Dec 1, 2025) — https://finance.yahoo.com/news/tutor-intelligence-raises-34-million-140000721.html  
8. Tutor Intelligence press page (links to coverage incl. USV/Series A) — https://tutorintelligence.com/press  
9. Skild AI robotics foundation model milestone (registry/summary source) — https://siliconvalleyinvestclub.com/companies/skild-ai  
10. Dropzone AI primary press release: **$37M Series B** (July 17, 2025) — https://www.dropzone.ai/press-release/dropzone-ai-37m-series-b-funding-ai-soc-agents  
11. Wikipedia: Manus launch (March 6, 2025) — https://en.wikipedia.org/wiki/Manus_(AI_agent)  
12. Info-Tech Research Group (independent research write-up on Manus) — https://www.infotech.com/research/assessing-manus-the-future-of-agentic-ai  
13. Hippocratic AI press page (links to “raises $126M series C…”, Nov 3, 2025) — https://hippocraticai.com/press  
14. SiliconANGLE / independent coverage of $126M Series C (Nov 3, 2025) — https://siliconangle.com/2026/01/15/higgsfield-raises-80m-1-3b-valuation-scale-ai-video-platform *(Note: this source is for Higgsfield in the snippet; not used for Hippocratic AI claims here. Replace with the correct Hippocratic AI-specific independent article if you want me to refine this entry with better-matched sources.)*  
15. McGuireWoods legal/industry alert referencing FDA clearance posting/letter date (Dec 23, 2025) — https://www.mcguirewoods.com/client-resources/alerts/2026/7/a-pathway-for-clinical-ai-developers-opens-fda-clears-first-software-as-a-medical-device-with-patient-facing-llm  
16. IntuitionLabs analysis referencing FDA decision posting/clearance details — https://intuitionlabs.ai/articles/fda-clears-first-llm-medical-device  
17. Eavor primary press release: “First Electricity Production at Geretsried Site” (Dec 4, 2025) — https://eavor.com/press-releases/eavor-technologies-achieves-first-electricity-production-at-geretsried-site  
18. IEEE Spectrum (independent via Canary Media) on advanced geothermal milestone — https://spectrum.ieee.org/advanced-geothermal-germany  

> **Data quality note:** One Hippocratic AI independent citation in the tool output appeared mismatched (it referenced Higgsfield). If you want, I can run a follow-up targeted search to add a correct Hippocratic AI-independent 2025 article (primary + independent, both explicitly about Hippocratic AI Series C).

In [29]:
#Run it
if research_agent is None:
	raise RuntimeError("Research agent is not initialized. Configure the API keys and rerun its setup cell.")

response_2 = run_research(research_agent, prompt_2)["answer"]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [30]:
display(Markdown(response_2))


## Executive summary (researched for calendar year 2025: 1 Jan–31 Dec 2025)
This overview identifies 8–12 **emerging** AI startups (excluding mature megacap tech/very late-stage public giants) that showcased meaningful **technical and/or commercial innovation** with **notable 2025 milestones**. Coverage spans foundation-model infrastructure, enterprise AI agents, healthcare, scientific discovery, robotics/perception, and climate/energy.

> **Research performed:** 2026-11-02.

> **Important limitation:** Public information for some 2025 milestones (especially funding amounts/dates) is incomplete or behind paywalls. Where reliable 2025 funding data could not be verified from a primary or credible independent source, it is omitted rather than guessed.

---

## Comparison table (2025 milestones)
| Startup | HQ country | Sector | Product/platform (what it solves) | Notable 2025 milestone |
|---|---|---|---|---|
| **Lila Sciences** | United States (company described as US-based in coverage) | Scientific discovery / scientific “superintelligence” | AI platform + autonomous “AI science factories” to generate hypotheses, design experiments, run them, and iterate for life sciences | Announced **$200M seed (Mar 2025)** for autonomous research at scale [1][2] |
| **Periodic Labs** | United States (SF listed in coverage) | Scientific discovery / autonomous labs | “AI scientists” where robots run experiments, collect data, iterate | Announced **$300M seed (Sep 30, 2025)** to automate science [3] |
| **AI Proteins** | United States | Healthcare / protein therapeutics | De novo designed **miniprotein** therapeutics using AI + high-throughput production | Announced **$41.5M Series A (Nov 19, 2025)** [4] |
| **Orbital Therapeutics** | United States | Healthcare / RNA therapeutics (AI-enabled R&D) | Circular RNA platform for in vivo CAR-T and other RNA therapeutics | **Bristol Myers Squibb acquisition announced (Oct 10, 2025)** for $1.5B (M&A milestone) [5] |
| **Corti** | Denmark (HQ described as Copenhagen) | Healthcare enterprise AI (specialized foundation models) | Healthcare-trained foundation models to power clinical/administrative AI with better healthcare alignment than generic LLMs | **Added UpToDate** to its foundation-model approach (partnership/expansion; 2025 reporting) [6] and launched healthcare foundation models framework (Jan 2025) [7] |
| **Regie.ai** | United States | Enterprise AI agents (sales workflow) | Agentic sales prospecting platform (“RegieOne”) that decides next actions in workflows | Announced **$30M Series B (Feb 26, 2025)** and launch push for RegieOne [8] plus patent milestone **Oct 7, 2025** [9] |
| **Apptronik** | United States (Austin) | Robotics / embodied AI | Humanoid robots and robotic systems for human-centered industrial tasks | Announced **$403M Series A close (Mar 18, 2025)** and large scaling momentum [10] |
| **KoBold Metals** | United States (San Francisco-based in coverage/brand reporting) | Climate/energy minerals via AI | AI-powered exploration for critical minerals needed for clean energy + electrification | Announced **$537M Series C (Jan 2025)** to expand critical mineral exploration [11] |
| **Fervo Energy** | United States (Houston-based in reporting) | Climate/energy (enhanced geothermal) | Next-gen geothermal development to deliver firm clean power | Announced/closed **$462M Series E (Dec 10, 2025)** (company press + reporting) [12] |

> The table is intentionally selective: it mixes “agentic/enterprise,” “scientific discovery,” “healthcare,” “robotics/perception,” and “climate/energy.”  
> I did **not** add “AI infrastructure inference” or “AI agent OS” startups where the most prominent 2025 launch/funding dates were either unverifiable or fell outside 2025 (e.g., Inferact’s main disclosed seed date appears in 2026, not 2025) based on available evidence [observed in searches].

---

## Detailed startup profiles (8–12 total; evidence-driven)

### 1) Lila Sciences
- **HQ country:** United States (coverage describes the company as US-based) [1]  
- **Founding year:** Not reliably verified in the sources retrieved for this run (omitted).
- **Product/platform & problem:**
  - Builds a “scientific superintelligence platform” working with **autonomous laboratories** to scale experimentation across life sciences, aiming to reduce friction between AI predictions and real physical validation. [1]
- **What’s technically/commercially innovative:**
  - “Generate hypotheses → design experiments → run them → learn → iterate” loop using autonomous lab infrastructure, targeted at the reproducibility and throughput bottlenecks in R&D. [2]
- **Notable 2025 milestone:**
  - **Seed funding: $200M** announced **Mar 2025**. [2]
- **Funding (2025, only if verifiable):**
  - **$200M seed**, announced **Mar 2025** (as reported by independent healthcare/biopharma press and other coverage). [1][2]
- **Primary source + independent source:**
  - Primary/near-primary: Altitude Life Science Ventures post referencing Lila’s $200M seed (publication date visible in retrieved result). [2]
  - Independent: BioPharma Dive reporting on the $200M seed and the autonomous lab/scientific superintelligence pitch (published **Mar 12, 2025**). [1]

**Why it’s worth watching:** If their autonomous science loop reaches scale across multiple life-science domains, it could represent a durable advantage versus “model-only” approaches by tightening the hypothesis-to-validated-result pipeline. The 2025 seed size signals investor belief in executing on that loop, not only publishing research. [1][2]

---

### 2) Periodic Labs
- **HQ country:** United States (SF referenced) [3]
- **Founding year:** Not reliably verified in sources retrieved (omitted).
- **Product/platform & problem:**
  - Aims to **automate scientific discovery** by creating labs where robots do experiments, collect data, and iterate—learning from outcomes over repeated cycles. [3]
- **What’s technically/commercially innovative:**
  - Focuses on the “closed loop” of **AI + robotics + experimental iteration**, rather than pure literature-based discovery.
- **Notable 2025 milestone:**
  - **$300M seed** announced **Sep 30, 2025**. [3]
- **Funding (2025, only if verifiable):**
  - $300M seed (date tied to the TechCrunch report). [3]
- **Primary source + independent source:**
  - Primary/close: Wilson Sonsini advisory note describing the **$300M seed** announcement date details. [4]
  - Independent: TechCrunch (published **Sep 30, 2025**). [3]

**Why it’s worth watching:** “AI scientists” in production require nontrivial engineering (robotic control, lab automation, data capture, safety/QA, and experimental design). The 2025 seed suggests they are betting on end-to-end automation with enough capital to prototype and iterate fast. [3][4]

---

### 3) AI Proteins
- **HQ country:** United States (Boston referenced) [4]
- **Founding year:** Not reliably verified in retrieved sources (omitted).
- **Product/platform & problem:**
  - De novo designed **miniprotein therapeutics**: AI helps design purpose-built proteins; the company pairs this with **high-throughput protein production** to advance candidates. [4]
- **What’s technically/commercially innovative:**
  - Targets a newer therapeutic modality where structured design + scalable production are key; the milestone includes platform improvements described alongside funding. [4]
- **Notable 2025 milestone:**
  - Announced **$41.5M Series A** on **Nov 19, 2025**. [4]
- **Funding (2025):**
  - $41.5M Series A (Nov 19, 2025). [4]
- **Primary source + independent source:**
  - Primary: Business Wire release from AI Proteins. [4]
  - Independent source: Not retrieved in the limited set for this run (only Business Wire was verified here).

**Why it’s worth watching:** The market signal is a step beyond “AI design demos” into scaling production and packaging a therapeutics pipeline around that capability—an area where execution and wet-lab throughput matter as much as modeling. [4]

---

### 4) Orbital Therapeutics
- **HQ country:** United States (Cambridge, MA referenced in coverage) [5]
- **Founding year:** Not reliably verified in retrieved sources (omitted).
- **Product/platform & problem:**
  - Circular RNA therapeutic platform; includes in vivo CAR-T (OTX-201) aimed at treating cancers and autoimmune diseases. [5]
- **What’s technically/commercially innovative:**
  - Platform strategy emphasizing **in vivo** RNA engineering/delivery (as described in acquisition coverage) with potential to broaden targeting beyond common constraints in RNA therapeutics. [5]
- **Notable 2025 milestone:**
  - **Acquisition announced by Bristol Myers Squibb (Oct 10, 2025)** for **$1.5B** in cash. [5]
- **Funding (2025):**
  - Not stated for 2025 in retrieved sources; this profile uses the 2025 M&A milestone instead.
- **Primary source + independent source:**
  - Primary: Not retrieved as a BMS press release in this run; acquisition details are supported by independent coverage.
  - Independent: BioPharma Dive article about Orbital’s earlier Series A (context) and acquisition-related facts; plus additional independent listing in retrieved results for the acquisition. [5]

**Why it’s worth watching:** A strategic acquisition in 2025 can indicate that (a) the platform reached a credible scientific/clinical stage and (b) the buyer sees differentiation that may be hard to replicate quickly. [5]

---

### 5) Corti
- **HQ country:** Denmark (Copenhagen referenced) [6]
- **Founding year:** Not reliably verified in retrieved sources (omitted).
- **Product/platform & problem:**
  - Healthcare-focused “foundation models” and infrastructure to power healthcare reasoning and reduce administrative burden, with emphasis on healthcare-data-only training to improve accuracy/compliance. [7]
- **What’s technically/commercially innovative:**
  - “Specialized foundation models” built exclusively on healthcare data (positioned as safer/more reliable than generic LLMs for healthcare tasks). [7]
- **Notable 2025 milestone(s):**
  - **UpToDate integration/adding UpToDate** to its foundation-model approach was reported in 2025 coverage. [6]
  - **Healthcare AI foundation model approach** announced early 2025 (Jan 28, 2025 PRNewswire). [7]
- **Funding (2025):**
  - Funding amount not retrieved/verified in this run for Corti; omitted.
- **Primary source + independent source:**
  - Primary: PRNewswire release on Corti launching specialized healthcare AI infrastructure (Jan 28, 2025). [7]
  - Independent: Fierce Healthcare coverage of Corti adding UpToDate (2025). [6]

**Why it’s worth watching:** Healthcare is a domain where “general capability” is not enough—governance, provenance, and clinical safety constraints matter. Corti’s 2025 partnership/integration signals traction in embedding healthcare knowledge sources into its foundation-model stack. [6][7]

---

### 6) Regie.ai
- **HQ country:** United States (San Francisco in PRNewswire) [8][9]
- **Founding year:** Not reliably verified in the retrieved sources (omitted).
- **Product/platform & problem:**
  - Agentic AI platform for **sales prospecting** (“RegieOne”), designed to automate enrichment, prioritization, and outbound execution while using an agent framework to decide next actions. [8][9]
- **What’s technically/commercially innovative:**
  - Patent milestone claims “next best action” decisioning in workflows—aiming to move beyond static sequences to dynamic action selection in enterprise settings. [9]
- **Notable 2025 milestone(s):**
  - **$30M Series B** announced **Feb 26, 2025**, tied to RegieOne launch scaling. [8]
  - **U.S. Patent No. 12346822B2 awarded** announced **Oct 7, 2025**. [9]
- **Funding (2025):**
  - $30M Series B (Feb 26, 2025). [8]
- **Primary source + independent source:**
  - Primary: PRNewswire / company announcement about Series B and RegieOne. [8]
  - Independent: PRNewswire also serves as a direct source; an additional independent outlet was not retrieved in this run for the patent/round. (The evidence is at least primary-corroborated by the patent announcement itself.) [9]

**Why it’s worth watching:** Enterprise agents compete on controllability, integration, and IP/protection of workflow decisioning. The combination of a 2025 funding round + a granted patent suggests both commercialization and defensibility were active in 2025. [8][9]

---

### 7) Apptronik
- **HQ country:** United States (Austin, TX) [10]
- **Founding year:** Not reliably verified in retrieved sources (omitted).
- **Product/platform & problem:**
  - Humanoid robots and robotics systems for **human-machine collaboration** in industrial settings. [10]
- **What’s technically/commercially innovative:**
  - Scaling embodied robotics: combining humanoid platform engineering with real-world deployments (as evidenced by large funding and continued scaling messaging in 2025 coverage).
- **Notable 2025 milestone:**
  - Announced **oversubscribed $403M Series A close** on **Mar 18, 2025**. [10]
- **Funding (2025):**
  - $403M Series A close (Mar 18, 2025), following an earlier $350M portion described in the same source. [10]
- **Primary source + independent source:**
  - Primary/near-primary: Apptronik’s news collection post about closing additional Series A funding. [10]
  - Independent: Not retrieved/verified in this run for the Mar 18 closure specifically.

**Why it’s worth watching:** Humanoid robotics is capital intensive, and 2025 funding at this scale suggests progress toward manufacturing/field deployment rather than purely lab prototypes. [10]

---

### 8) KoBold Metals
- **HQ country:** United States (brand origin often SF; verified as US company in prior reporting though not exhaustively sourced here)
- **Founding year:** Not reliably verified in retrieved sources (omitted).
- **Product/platform & problem:**
  - AI-powered critical mineral exploration to accelerate discovery and diversification of minerals needed for clean energy and electrification. [11]
- **What’s technically/commercially innovative:**
  - “AI-powered exploration” positioned to reduce the slow, expensive cycle of getting new mines to production—commercial innovation aimed at speeding exploration-to-development.
- **Notable 2025 milestone:**
  - Announced **$537M Series C** on **Jan 1 / early Jan 2025 timeframe** (X post with date visible) and expanded exploration. [11]
- **Funding (2025):**
  - $537M Series C. [11]
- **Primary source + independent source:**
  - Primary-ish: KoBold’s own X post thread with funding details and date shown (Jan 1, 2025). [11]
  - Independent: Not retrieved/verified for the exact $537M in this run beyond that primary-ish source.

**Why it’s worth watching:** Clean energy supply chains depend on mineral discovery at scale. An AI-first exploration company is worth monitoring for whether it can translate modeling improvements into faster drilling/asset conversion—something investors signaled by backing with very large capital. [11]

---

### 9) Fervo Energy
- **HQ country:** United States (Texas-based referenced in coverage) [12]
- **Founding year:** Not reliably verified in retrieved sources (omitted).
- **Product/platform & problem:**
  - Next-generation geothermal development using enhanced geothermal systems to deliver **firm clean power** to the grid. [12]
- **What’s technically/commercially innovative:**
  - Commercial scaling of geothermal (timelines and capacity buildout stated in the company’s financing press release and reporting).
- **Notable 2025 milestone:**
  - **$462M Series E** announced/closed **Dec 10, 2025** to accelerate buildout of Cape Station and other projects. [12]
- **Funding (2025):**
  - $462M Series E (Dec 10, 2025). [12]
- **Primary source + independent source:**
  - Primary: Fervo Energy newsroom/press release entry describing the financing and Cape Station delivery schedule. [12]
  - Independent: Energy Capital article reported the round and date (Dec 10, 2025). [13]

**Why it’s worth watching:** Geothermal scale-up is hard; in 2025, the market rewarded companies with credible commercialization pathways for firm renewables. A large Series E suggests Fervo’s technical/commercial progress was sufficiently de-risked for investors to fund expansion. [12][13]

---

## 3–5 particularly interesting innovations (explained)
1. **Closed-loop “AI scientist” automation (Periodic Labs, Lila Sciences)**  
   Instead of using AI to draft hypotheses only, these startups aim for **repeated experiment iteration** using lab automation/robots plus AI-driven learning loops—tackling the “AI-to-physical validation” bottleneck. Evidence: Periodic Labs’ goal explicitly described as automating discovery with robotic experiments and iteration; Lila’s platform described as paired with autonomous labs for hypothesis→experiment→learn cycles. [3][2]

2. **Domain-specialized “healthcare foundation models” (Corti)**  
   Corti positions its approach around healthcare-only training and healthcare-native infrastructure, aiming to improve clinical reliability and reduce time spent correcting AI outputs. Evidence: Corti’s Jan 2025 PRNewswire release and 2025 Fierce Healthcare coverage of UpToDate integration. [7][6]

3. **Agentic workflow “next best action” (Regie.ai)**  
   Regie.ai’s 2025 milestone includes a granted U.S. patent for agentic decisioning in workflows (sales prospecting context, but patent described as framework applicable across business functions). This reflects a shift from “LLM chat” toward **decisioning + action execution** in enterprise processes. [9]

4. **Embodied robotics scaling with large capital infusion (Apptronik)**  
   Apptronik’s 2025 multi-hundred-million Series A close is notable because humanoids require substantial engineering + manufacturing and long iteration cycles. The 2025 round indicates progress toward commercialization rather than only research. [10]

5. **AI-first critical mineral exploration + firm power geothermal as “climate AI” bets (KoBold, Fervo)**  
   Both emphasize translating AI/technology into deployment: KoBold focuses on accelerating mineral discovery, while Fervo focuses on scaling geothermal to provide firm clean power. Evidence includes KoBold’s $537M Series C announcement and Fervo’s $462M Series E press release + independent reporting. [11][12][13]

---

## Funding and market patterns (what 2025 data suggests)
- **Capital concentration in platforms that can “close the loop”:** Scientific discovery startups (Lila Sciences, Periodic Labs) and enterprise agents (Regie.ai) received large rounds tied to execution narratives beyond models alone. [2][3][8]
- **Healthcare remains a “trust + integration” battleground:** Corti’s 2025 partnership/integration with clinical knowledge (UpToDate) indicates emphasis on embedding authoritative sources and domain training into foundation-model workflows. [6][7]
- **Climate/energy AI is increasingly linked to deployment bottlenecks:** Mineral discovery and geothermal scaling received very large rounds in 2025, reflecting investor focus on real-world capacity creation rather than purely analytical tools. [11][12][13]
- **Robotics funding is huge but milestones are fewer:** Apptronik’s 2025 funding suggests investors are underwriting long build cycles when there’s credible scaling momentum. [10]

### Limitations in public data (important)
- **Incomplete verifiable 2025-only funding:** For some startups, reliable 2025 funding amounts/dates were not found in accessible sources during this run; per your rules, I omitted unverified numbers.
- **M&A vs fundraising:** Some “milestones” are acquisitions rather than funding; both are valid but represent different stages of “emerging.”
- **Selection bias toward companies with strong press coverage:** Niche/stealth startups may have 2025 milestones that were not publicly documented in the sources retrieved here.

---

## Conclusion
In 2025, the most compelling emerging AI startup innovations were concentrated in approaches that **combine models with real-world systems**—autonomous lab iteration, healthcare-native specialized foundation models, agentic workflow decisioning, embodied robotics scaling, and deployment-oriented climate/energy technologies. The biggest 2025 signals in public sources were (a) **large early-to-mid rounds** for closed-loop scientific and climate platform bets and (b) **enterprise adoption pathways** (integrations, patents, platformization) that aim to move beyond pilots toward operational use. [2][3][6][8][9][11][12]

---

## Sources (published during/visible for 2025 where possible; URLs direct)
1. BioPharma Dive — “Flagship startup raises $200M in pursuit of ‘scientific superintelligence’” (Published **Mar 12, 2025**)  
   https://www.biopharmadive.com/news/lila-flagship-ai-superintelligence-startup-seed/742213  
2. Altitude Life Science Ventures — Post referencing Lila Sciences $200M seed (Retrieved source with publication date visible in result)  
   https://www.altitudelsv.com/single-post/us-ai-biotech-startup-lila-sciences-raises-235m-valued-at-1-2-billion  
3. TechCrunch — “Former OpenAI and DeepMind researchers raise whopping $300M seed to automate science” (Published **Sep 30, 2025**)  
   https://techcrunch.com/2025/09/30/former-openai-and-deepmind-researchers-raise-whopping-300m-seed-to-automate-science  
4. Wilson Sonsini — “Periodic Labs on $300 Million Seed Round” (Client highlight page; includes date details)  
   https://www.wsgr.com/en/insights/wilson-sonsini-advises-periodic-labs-on-dollar300-million-seed-round.html  
5. Multiples (VC data aggregator) — Orbital acquisition details (used for M&A fact in this run)  
   https://multiples.vc/private-comps/orbital-therapeutics  
6. Fierce Healthcare — “Corti AI adds UpToDate to its foundation model” (2025 coverage; accessed via retrieved snippet)  
   https://www.fiercehealthcare.com/ai-and-machine-learning/exclusive-corti-ai-adds-uptodate-its-foundation-model  
7. PRNewswire — Corti launches specialized healthcare AI infrastructure (Published **Jan. 28, 2025**)  
   https://www.prnewswire.com/news-releases/corti-launches-specialized-healthcare-ai-infrastructure-challenging-industry-misuse-of-general-purpose-models-302360834.html  
8. PRNewswire/CMSWire syndication — Regie.ai raises $30M Series B; launching RegieOne (Published **Feb. 26, 2025** in the PR text)  
   https://www.cmswire.com/the-wire/regieai-raises-30m-series-b-to-scale-ai-powered-sales-strategies-introducing-regieone-platform-for-smarter-prospecting  
9. PRNewswire — Regie.ai awarded U.S. Patent No. 12346822B2 (Published **Oct. 7, 2025**)  
   https://www.prnewswire.com/news-releases/regieai-secures-foundational-us-patent-for-ai-agent-technology-years-ahead-of-the-market-302576918.html  
10. Apptronik — “Apptronik closes additional Series A funding…” (Mar 18, 2025 in retrieved result)  
   https://apptronik.com/news-collection/apptronik-closes-additional-series-a-funding  
11. KoBold Metals — X post with $537M Series C details (Date shown as **Jan 01, 2025**)  
   https://x.com/KoBold_Metals/status/1874833550949970328  
12. Fervo Energy newsroom — “Fervo Energy Raises $462 Million Series E…” (press release dated **Dec 10, 2025** in retrieved result)  
   https://fervoenergy.com/newsroom-2-2-2-2-2  
13. Energy Capital — coverage of Fervo Energy closes $462M Series E (Dated **Dec 10, 2025** in retrieved result)  
   https://energycapitalhtx.com/fervo-energy-series-e  

If you want, I can rerun the search with a stricter constraint of **“primary source for each item”** (every funding + milestone backed by company/filing/regulator documents) and produce an updated list where any item lacking primary-source evidence is replaced.

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




## Exercise 3: Compare Two Tech Products
Goal: Compare OpenAI ChatGPT Team and Anthropic Claude Pro in terms of features, pricing, practical value, and independent user reviews.

Important: Product names, plans, and prices can change. The prompt requires the agent to explain the historical ChatGPT Team name and identify any current successor or renamed offering rather than silently treating different plans as identical.

In [31]:
# Your prompt here
prompt_3 = """
Prepare an evidence-based comparison of OpenAI ChatGPT Team and Anthropic
Claude Pro.

Research the latest verifiable information available on the date you run
this prompt. The comparison must cover:

1. Product positioning and intended users.
2. Core model access and capabilities, including reasoning, coding,
   file analysis, context limits, multimodality, and web/research features
   where available.
3. Collaboration and administration: team workspaces, user management,
   security, privacy, data controls, and integrations.
4. Pricing: list price, billing period, minimum seats if applicable,
   regional/currency caveats, and whether taxes are included when stated.
5. Usage limits and important restrictions. Do not imply unlimited usage
   unless official documentation explicitly supports it.
6. Independent reviews and user feedback: summarize recurring strengths
   and weaknesses from credible reviewers or review platforms. Separate
   anecdotal user comments from systematic review evidence.
7. Best-fit scenarios: individual professional, small team, developer,
   researcher, and organization.

Historical/product-name clarification:
- ChatGPT Team may have been renamed or superseded. Explain the relationship
  between the historical ChatGPT Team plan and the current relevant OpenAI
  business plan, using official sources.
- Do not compare ChatGPT Team with Claude Pro as if they necessarily target
  the same type of customer. Explain the individual-versus-team distinction.
- If presenting historical 2025 pricing, label it separately from current pricing.

Output format: clean Markdown with:
- Executive summary
- Side-by-side comparison table
- Pricing table with currency and checked date
- Review and reputation summary
- Which product is better for which user?
- Limitations and caveats
- Final balanced verdict
- Sources (numbered, direct URLs)

Prioritize official OpenAI and Anthropic documentation for features and prices.
Use independent sources for reviews. State the date on which the research
was performed. If a fact cannot be verified, say so instead of guessing.
"""

# Run Exercise 3.
if research_agent:
    response_3 = run_research(research_agent, prompt_3)
    display_run("Exercise 3 — ChatGPT Team vs Claude Pro", response_3)
else:
    print("Configure API keys and initialize the agent to run Exercise 3.")

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


## Exercise 3 — ChatGPT Team vs Claude Pro
Tools called: ['web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search']
Model calls: 2
Tokens (input/output/total): 28514 4694 33208


## Executive summary (checked **2026-11-02**)
- **OpenAI “ChatGPT Team”** has been **renamed to “ChatGPT Business”** (OpenAI states the features and pricing didn’t change with the naming update). Use this plan when you need **a shared workspace with admin/billing controls** for a team. [1][2][3]
- **Anthropic Claude Pro** is an **individual subscription** that upgrades you to higher usage capacity (e.g., “5x more usage” of Claude 2 at launch; current exact caps are plan-dependent and dynamic). It’s not a team-admin product. [4]
- If your priority is **team administration + shared workspace**, **ChatGPT Business** is the direct analogue to Claude’s team offering (not Claude Pro). [2]
- If your priority is **personal pro usage for coding/research/multimodal + higher caps**, **Claude Pro ($20/month US)** is straightforward; you’ll typically be trading away team governance features you’d get on OpenAI’s business plan. [4]

---

## Side-by-side comparison (ChatGPT Team → ChatGPT Business vs Claude Pro)

> **Important scope clarification:** Anthropic’s “Claude Pro” is an *individual* plan; OpenAI’s “ChatGPT Team” (now “ChatGPT Business”) is a *team workspace* plan with centralized administration. They are not direct “apples-to-apples” in customer targeting. [2][4]

| Dimension | OpenAI: **ChatGPT Team (renamed to ChatGPT Business)** | Anthropic: **Claude Pro** |
|---|---|---|
| **1) Positioning & intended users** | Self-serve plan for organizations needing a **shared workspace**, **centralized billing**, and **admin controls** for team users. [2] Also: OpenAI explicitly says “ChatGPT Team is now called ChatGPT Business.” [1] | Individual “Pro” subscription for higher usage. Anthropic’s launch post frames it for personal subscribers and “productivity” tasks. [4] |
| **2) Core model access & capabilities** | **Model/tool access varies** by what OpenAI enables in the Business workspace and what OpenAI considers “frontier model” access; OpenAI states the plan provides access to advanced models and tools (e.g., **Advanced Data Analysis** was mentioned for Team at launch). [1] Also, ChatGPT’s model lineup evolves over time (e.g., GPT‑4 replaced by GPT‑4o in ChatGPT on April 30, 2025 per release notes). [5] | Anthropic positions Claude Pro as **higher usage** for its latest Claude model(s). At Pro launch, Anthropic states Pro provides “5x more usage” of **Claude 2**. [4] Current model availability/caps are not fully enumerated in the sources retrieved here beyond the general “usage” and that newer models support larger contexts; for exact current per-model context/caps, you should consult Anthropic’s in-product “Usage”/model selector and help docs. *(Not fully verifiable from the sources gathered in this run.)* |
| **Reasoning** | Not quantified in the sources retrieved here for Business vs other tiers; Business includes access to advanced models and tools, but **exact reasoning-tier message caps** are not stated in the retrieved sources. [1][2] | Claude Pro is described as increasing “usage” of Claude; **exact reasoning capacity/caps** are dynamic. [4] |
| **Coding** | OpenAI states Business/T​eam includes **Codex access depending on seat type**, and the plan includes Codex in some form (baseline access is stated). [2] | Claude Pro supports coding workflows (Anthropic’s launch post references “iterating further on coding projects”). [4] (Exact coding-model routing/caps not fully specified in retrieved sources.) |
| **File analysis (uploads)** | OpenAI provides file upload capability in ChatGPT. Official help docs mention upload caps in general ChatGPT usage (e.g., **up to 80 files every 3 hours**; also plan-dependent file limits). Retrieved sources here did not fully isolate Business-specific upload caps. [6] | Anthropic provides file upload capability via Claude app: official help doc confirms file upload support and limitations for “Upload files to Claude” (e.g., **30MB per project file**, **unlimited count if total fits context**, multimodal PDFs for certain sizes). [7] |
| **Context limits** | For Business/ChatGPT Enterprise, OpenAI help says **Enterprise uses a maximum context window of 128k tokens**; it also discusses “stuffed tokens” for uploads. The retrieved source explicitly mentions **Enterprise**, not Business. [8] (So Business context size is not directly verified here.) | Claude help doc states: on paid plans, **newest models support up to 1M token context window**, and mentions other token sizes for other models. [9] |
| **Multimodality** | GPT‑4o is natively multimodal in ChatGPT per release notes; Business includes access to current ChatGPT models depending on plan. [5] | Claude models support multimodal inputs; Anthropic’s file upload/help indicates multimodal PDF handling. [7] |
| **Web / research features** | Not fully verified for Business specifically in the sources retrieved here (we only have general community discussion and not official product docs in this run). **Do not treat community posts as proof.** | Not fully verified for Claude Pro in the sources retrieved here from official docs (we have some “web search tool” third-party explanations, but those are not primary sources). For reliable verification, use Anthropic help docs for “web search” availability by plan. *(Not fully verified in this run.)* |

---

## 3) Collaboration & administration (workspaces, users, security/privacy, integrations)

| Category | OpenAI: ChatGPT Business (Team → Business) | Anthropic: Claude Pro |
|---|---|---|
| **Team workspaces** | Designed for a **shared ChatGPT workspace** for teams. [2] | No team workspace/admin features described in the retrieved sources; Claude Pro is an individual subscription. [4] |
| **User management / roles** | OpenAI help describes Business workspace roles such as **Owner/Admin/Analytics Viewer**, with Owners having identity management and workspace configuration; Members use ChatGPT but lack admin privileges. [10] | Not applicable to Pro (individual). [4] |
| **Security controls** | OpenAI positions Business as secure collaboration with admin controls; plus “Admin Console” governance is part of the plan. [2][10] | Not covered for Pro in the retrieved sources beyond general privacy-policy behavior; exact enterprise-grade controls are not verifiable here. |
| **Data privacy / training** | OpenAI states for Team (now Business): “We do not train on your business data or conversations, and our models don’t learn from your usage” (as with ChatGPT Enterprise). [1] Also OpenAI states Business data privacy commitments on its business-data page: “We don’t train our models on your organization’s data by default.” [11] | Anthropic’s Pro privacy/training behavior is **not confirmed by official Anthropic sources in the retrieved set**. Some third-party pages discuss opt-in/out and retention, but I cannot treat them as authoritative here. *(So: Claude Pro privacy/training specifics are not verifiable in this run.)* |
| **Integrations** | OpenAI provides “Apps” and admin controls over connected apps/plugins. Retrieved source covers **admin controls for plugins and apps** in Admin Console. [12] | Claude Pro integration list not verified in the retrieved sources set. |

---

## 4) Pricing (list price, billing period, seats, tax notes)

### OpenAI: ChatGPT Business (formerly ChatGPT Team)
OpenAI help center states: [13]
- **Monthly plan:** **$25 per user per month**
- **Annual plan:** **$20 per user per month**, billed annually
- **Minimum purchase:** **2 standard seats**

Also: OpenAI help notes a **pricing structure change**: “As of April 2, 2026, we’ve changed… reduced the price of standard … by USD $5 per month” (with region/currency caveat). [13]

### Anthropic: Claude Pro
Anthropic launch post states (US/UK): [4]
- **$20/month (US)** and **£18/month (UK)**

> **Taxes:** Neither OpenAI’s retrieved pricing snippet nor Anthropic’s launch post explicitly states whether taxes are included in the displayed prices. So tax inclusion is **not verifiable from the sources retrieved here**.

#### Pricing table (currency + checked date)

| Product | Price (list) | Billing period | Minimum seats | Tax included? | Checked on |
|---|---:|---|---:|---|---|
| **ChatGPT Business (Standard seat)** | **$25/user/month** (monthly) | Monthly | **2 users (minimum)** | Not stated in retrieved official snippets | 2026-11-02 |
| **ChatGPT Business (Standard seat)** | **$20/user/month** (annual) | Billed annually | **2 users (minimum)** | Not stated in retrieved official snippets | 2026-11-02 |
| **Claude Pro** | **$20/month (US)** | Monthly | N/A (individual) | Not stated in retrieved official snippets | 2026-11-02 |

---

## 5) Usage limits & important restrictions (what’s verifiable)

### ChatGPT Business / (Team)
- **Message caps / rate limits:** not reliably captured from official docs in the retrieved sources for this run; some community discussion exists but is not official documentation, so I’m not using it as evidence.
- **File upload limits:** OpenAI help indicates **up to 80 files every 3 hours** for uploads (in general ChatGPT); it also references plan-dependent limits and that peak-hour limits may drop. [6]  
  - Business-specific upload caps were **not isolated** from the retrieved sources here.
- **Context limits:** the retrieved official “uploaded file context” source explicitly describes **Enterprise** context window and uploaded token handling, not Business. [8]

### Claude Pro
- Anthropic’s Pro announcement provides **relative usage** (“5x more usage” of Claude 2 at launch). [4]
- Claude help indicates paid plans support large contexts (up to **1M tokens** depending on model), and also explains that Claude automatically manages long conversations via summarization near context limits. [9]
- **Exact Pro usage quotas (per 5-hour / weekly caps)** were not directly verified from official Anthropic help pages in the sources gathered in this run—so I’m not stating numeric caps for Pro beyond the “5x more usage” claim at launch. [4]

---

## 6) Independent reviews & user feedback (credible evidence only; separated from anecdotes)

### Method note
In the sources gathered here, independent reviews came from review platforms/community posts. However, for **Claude**, I found review-site material, and for **ChatGPT Business**, I found more community feedback than systematic review evidence. I therefore keep Claude review summaries stronger than ChatGPT review summaries.

### Claude Pro (review / reputation evidence)
- **Trustpilot:** One page shows **low rating** with complaints such as:
  - reliability issues (“states things it has not checked”)
  - frustration with **usage limits** / session caps disrupting work
  - billing/support dissatisfaction  
  This is a **systematic review-site signal**, not a single anecdote, but specific figures should be treated as platform-specific and time-varying. [14]
- **G2 (article summarizing reviews):** An article reports relatively high satisfaction (e.g., overall rating shown as **4.4/5** and highlights strengths like ease of use and coding/debugging, plus long context handling). This is still secondary content summarizing user feedback, not a primary dataset. [15]

**Recurring strengths (from review summaries):**
- Ease of use / good workflow for knowledge work
- Strong coding/debugging assistance
- Handling long documents/context continuity  
(From G2 review summary.) [15]

**Recurring weaknesses (from review-site evidence):**
- Limits/session throttling affecting sustained workflows
- Hallucination/verification concerns reported
- Billing/support complaints  
(From Trustpilot.) [14]

### ChatGPT Business / Team
- In this run, the retrieved evidence for ChatGPT Team/Business reviews is mostly **OpenAI community forum threads** (individual posts, not a systematic review platform). Because you asked to “separate anecdotal user comments from systematic review evidence,” I’m not claiming broad consensus on strengths/weaknesses for ChatGPT Business from those threads alone.  
- I do have official documentation on collaboration/admin and privacy posture, so the comparison for ChatGPT leans on official sources, not review consensus. [1][2][11]

---

## Which product is better for which user?

| User profile | Better fit (most often) | Rationale |
|---|---|---|
| **Individual professional (single user)** | **Claude Pro** *or* OpenAI individual plans | Claude Pro is explicitly an individual subscription. [4] If you’re not needing workspace admin, Claude Pro avoids team governance complexity. |
| **Small team (2–10 people) needing shared workspace + controls** | **ChatGPT Business (Team → Business)** | OpenAI’s plan is built for shared workspace, centralized billing, and admin controls. [2][10] Claude Pro doesn’t provide that team governance (based on retrieved sources). |
| **Developers / coders** | Depends on workflow | OpenAI Business includes **Codex access depending on seat type**. [2] Anthropic notes Pro supports coding iterations. [4] For exact “which codemodel + how much” caps, official per-tier docs weren’t fully captured for both sides in this run. |
| **Researchers / heavy document analysis** | **Claude Pro** (often) for context headroom; **ChatGPT Business** (often) if you need team governance | Claude help indicates paid plans/newest models support up to **1M tokens** context. [9] OpenAI’s upload processing details are verified for Enterprise, not Business. [8] So “Claude for long-context research” is better supported. |
| **Organization / compliance-focused admin** | **ChatGPT Business** (and for stricter needs, Enterprise—if budget allows) | OpenAI provides admin roles and “don’t train on business data by default” claims for Business/Enterprise. [1][10][11] For Claude, Pro’s enterprise-grade controls weren’t verified here. |

---

## Limitations and caveats (important)
1. **This research is only as complete as the sources retrieved in this run.** For some requested areas (notably **exact numeric usage caps** for Claude Pro and **Business-specific message/file caps**), I could not verify from official docs within the gathered sources—so I did **not** guess.
2. **Model/capability availability changes over time.** OpenAI’s ChatGPT model lineup (e.g., GPT‑4 → GPT‑4o replacement) is documented, but what’s “included” in Business can shift with product updates. [5]
3. **Claude Pro privacy/training specifics** were not verified from official Anthropic sources in the retrieved set; I therefore did not assert training/retention behavior for Pro.
4. **Tax inclusion** for both products is not explicitly verified in the retrieved pricing snippets; treat list prices as “before tax unless your checkout says otherwise.”

---

## Final balanced verdict
- Choose **ChatGPT Business (formerly ChatGPT Team)** if you need a **team workspace** with **centralized billing, admin/user roles, and governance**, plus OpenAI’s documented posture of not training on your organization’s data by default. [1][2][10][11]
- Choose **Claude Pro** if you want an **individual “pro” tier** for **higher usage** and **large-context document work**, with official support that paid plans’ newest models can reach **up to 1M token context windows** (exact per-model availability/caps should be confirmed in-product). [4][9]
- They are not perfectly comparable because **Claude Pro is individual** while **ChatGPT Team/Business is team-admin**. For a “team analogue” to Claude, you’d need to compare OpenAI Business with **Anthropic’s team plan**—not Claude Pro. *(Not performed here because the prompt asked specifically for Claude Pro.)*

---

## Sources
1. OpenAI blog: **“Introducing ChatGPT Team”** (includes rename update: “ChatGPT Team is now called ChatGPT Business. Features and pricing haven’t changed.”) https://openai.com/index/introducing-chatgpt-team  
2. OpenAI Help Center: **“ChatGPT Business overview”** https://help.openai.com/en/articles/8792828-chatgpt-business-overview  
3. OpenAI release notes (model retirement context): **ChatGPT release notes** https://help.openai.com/en/articles/6825453-chatgpt-release-notes  
4. Anthropic news: **“Introducing Claude Pro”** https://www.anthropic.com/news/claude-pro  
5. OpenAI Help Center: **ChatGPT release notes** (GPT‑4 retired in ChatGPT; replaced by GPT‑4o) https://help.openai.com/en/articles/6825453-chatgpt-release-notes  
6. OpenAI Help Center: **“Uploading files and audio to ChatGPT”** https://help.openai.com/en/articles/8555545-uploading-files-and-audio-to-chatgpt  
7. Anthropic Help Center: **“Upload files to Claude”** https://support.claude.com/en/articles/8241126-upload-files-to-claude  
8. OpenAI Help Center: **“Optimizing file uploads in ChatGPT Enterprise”** (context window details for Enterprise) https://help.openai.com/en/articles/10029836-optimizing-file-uploads-in-chatgpt-enterprise  
9. Anthropic Help Center: **“How do usage and length limits work?”** https://support.claude.com/en/articles/11647753-how-do-usage-and-length-limits-work  
10. OpenAI Help Center: **“Managing members, seat types, and roles in ChatGPT Business”** https://help.openai.com/en/articles/8542216-managing-members-seat-types-and-roles-in-chatgpt-business  
11. OpenAI: **Business data privacy / commitments** https://openai.com/business-data  
12. OpenAI Help Center: **“Admin controls, security, and compliance for plugins and apps”** https://help.openai.com/en/articles/11509118-admin-controls-security-and-compliance-for-plugins-and-apps  
13. OpenAI Help Center: **“Managing billing and seats in ChatGPT Business”** https://help.openai.com/en/articles/8792536-managing-billing-and-seats-in-chatgpt-business  
14. Trustpilot: **Claude reviews page** https://www.trustpilot.com/review/claude.ai  
15. G2 review summary (secondary): **“My Claude AI Review (2026)”** https://learn.g2.com/claude-ai-review  

If you want, I can run a second pass focused specifically on **official, plan-specific usage caps** (Claude Pro and ChatGPT Business) by pulling the exact help-center/limits pages for each plan and extracting the current numeric quotas.

In [33]:
# Run it
if research_agent is None:
	raise RuntimeError("Research agent is not initialized. Configure the API keys and rerun its setup cell.")

response_3 = run_research(research_agent, prompt_3)["answer"]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [34]:
display(Markdown(response_3))


## Executive summary (checked **2026-11-02**)
- **OpenAI’s “ChatGPT Team” is renamed to “ChatGPT Business”** (no feature/limits change implied by the rename). The current comparison target for “ChatGPT Team” is therefore **ChatGPT Business**. [1][2]
- **Claude Pro** is an **individual** plan, while **ChatGPT Business** is a **team/workspace** plan. They are not perfectly comparable on target customer, even though both are paid subscriptions. [1][3]
- Both products include **multimodal capabilities** (file/image handling) and **coding features** within their ecosystems, but **their packaging differs**:
  - **ChatGPT Business** emphasizes **shared workspaces + admin controls** for teams and includes features like **Projects, Company Knowledge, Deep Research, and Codex**. [4]
  - **Claude Pro** emphasizes **higher personal capacity** and includes **Claude Code** and other pro-only workflows, but collaboration/admin is not the core of the plan. (Some “team” features exist only on Anthropic’s team tier, not Pro.) [5][6]

---

## Side-by-side comparison table

| Dimension | OpenAI **ChatGPT Business** *(formerly “ChatGPT Team”)* | Anthropic **Claude Pro** |
|---|---|---|
| **Positioning / intended users** | Self-serve **organizational/team** workspace with centralized management, shared work areas, and admin controls. [4][7] | **Individual professional** plan focused on larger capacity and pro features for one user. [6] |
| **Core access / model & capabilities** | Includes access to **ChatGPT, ChatGPT Work, and Codex**, plus tools like **Projects, Company Knowledge, ChatGPT Agent, Deep Research**. Included with standard seats; Premium has higher included usage. [4] | Pro is the consumer tier that unlocks **pro usage capacity** and includes capabilities such as **Claude Code** (coding assistant). (Specific model roster/limits are not fully verifiable from official docs in the sources retrieved.) [6] |
| **Reasoning & coding** | Includes **Codex** and agentic “Work/Deep Research” features; coding is supported via Codex within the ChatGPT ecosystem. [4] | **Claude Code** is part of Pro; useful for coding workflows. (Official usage-limit structure not fully pinned down in retrieved official sources.) [5][6] |
| **File analysis / uploads** | Projects support **file uploads**, with upload caps depending on plan. For Business/Enterprise/Edu: **40 files per project**; also only **10 files at the same time**. [8] | File analysis is available as part of Claude’s multimodal system, but **specific per-plan caps** (e.g., “N files”) were not verifiable from Anthropic official Pro sources in the retrieved set. |
| **Multimodality (images/files)** | ChatGPT Work/ChatGPT capabilities include multimodal support (e.g., vision; exact per-tier caps not fully itemized in retrieved official sources). Business includes access to the ChatGPT product suite. [4] | Claude supports **vision/image processing** (developer docs note image processing/handling, but Pro-specific limits/caps not fully pinned down in retrieved official Pro sources). [9] |
| **Web / research features** | ChatGPT Business overview includes **Deep Research**. [4] | Claude Pro supports browsing/research features in some form, but the **specific “web/research” tools included in Pro** were not fully verifiable from official Claude Pro pages in the retrieved sources. |
| **Collaboration & administration** | **Workspace** model with team membership; centralized **administration** and workspace-level controls. [7][4] | Pro is primarily **individual**; collaboration/admin is generally a **team tier** concern (Anthropic’s team offering), not Pro. [6] |
| **Security & privacy stance** | OpenAI states for Business: **no training on business data by default**; provides privacy/security commitments and limited access by authorized staff/contractors for abuse/legal/engineering purposes. [10][11] | Anthropic consumer privacy model includes a **training/retention choice** for Free/Pro/Max (vs. commercial plans with stricter guarantees). [12][13] |
| **Usage limits & restrictions** | Included access is **rate/usage-limited** with “workspace credits” to extend usage beyond included rate limits. [4][14] | Pro has usage caps (commonly described as time-window metering), but **official numeric caps** were not fully verifiable from the official Pro sources retrieved here. [6] |

---

## Historical/product-name clarification (ChatGPT Team → ChatGPT Business)

OpenAI explicitly states that:
- **“ChatGPT Team” was renamed to “ChatGPT Business” on August 29, 2025**, and it is a **name change only** with **no feature/limits/security change** implied by the rename. [1][2]

**Implication for this comparison:** when you ask for “ChatGPT Team,” the current corresponding OpenAI business plan to compare against Claude Pro is **ChatGPT Business**. [1][2]

---

## Pricing (USD) — with checked date: **2026-11-02**

> Note: Prices may vary by country/currency and taxes; where official sources explicitly mention taxes, I report it. Otherwise, I do not assume tax treatment.

### OpenAI — ChatGPT Business (formerly Team)
From OpenAI help center billing guidance:
- **Standard seat:** **$25/user/month** billed monthly **or $20/user/month** billed annually  
- **Premium seat:** price is listed in the Business overview page (see below)
- **Minimum:** **2 standard seats** required for standard billing guidance. [14]

From OpenAI Business overview:
- **Premium seat:** **$125/user/month** billed monthly **or $100/user/month** billed annually. [4]
- OpenAI also notes: **“pricing may vary depending on your country and currency type.”** [4]

#### Pricing table
| Plan | Price (USD) | Billing period | Minimum seats | Taxes included? |
|---|---:|---|---:|---|
| ChatGPT Business **Standard** | $25/user/month (monthly) or $20/user/month (annual) | Monthly or annual | 2 standard seats minimum | Not stated in retrieved OpenAI sources |
| ChatGPT Business **Premium** | $125/user/month (monthly) or $100/user/month (annual) | Monthly or annual | Not explicitly stated in retrieved sources (see note) | Not stated in retrieved OpenAI sources |

**Note on minimum seats:** OpenAI explicitly states a **minimum purchase of 2 standard ChatGPT seats** for standard billing guidance. [14] Whether Premium-only workspaces have the same minimum in the same exact wording wasn’t independently confirmed in the retrieved sources.

### Anthropic — Claude Pro
I did not retrieve an official Anthropic pricing page in the tool results here; the retrieved sources cite **$20/month** and **annual discount**, but they are **secondary**. Because you asked to prioritize official docs, treat the numbers below as **not fully verifiable from primary pricing documentation in the retrieved set**.

| Plan | Price (USD) | Billing period | Minimum seats | Taxes included? |
|---|---:|---|---:|---|
| Claude Pro | $20/month; often cited as $17/month when billed annually | Monthly or annual (secondary citation) | N/A (individual) | Not stated |

Sources retrieved for Claude Pro pricing used third-party summaries [6], so **I can’t mark this as “officially verified”** from Anthropic’s primary pricing page based on what was retrieved.

---

## Usage limits & important restrictions (what’s verifiable)

### ChatGPT Business — verifiable restrictions
- **Projects file upload caps:**  
  - Only **10 files can be uploaded at the same time**  
  - For **Business/Enterprise/Edu: 40 files per project**  
  - OpenAI also notes “Projects” can be unlimited in count (i.e., unlimited projects), but file caps apply per project. [8]
- **Usage limits extend via workspace credits:**  
  ChatGPT Business seats include baseline access and **included rate limits**, and you can **purchase workspace credits** to extend beyond included limits. [4][14]
- **Codex/workspace usage is metered:** OpenAI states codex usage beyond included rate limits can be extended via credits. [14]

### Claude Pro — what’s *not* fully verifiable here
- While third-party sources describe **time-window metering** and other caps for Claude Pro/Claude Code, I did **not** retrieve an official Anthropic Pro usage-limits article with explicit numeric quotas in the current evidence set. Therefore I do **not** state specific caps here beyond what’s directly supported by retrieved sources. [6]

---

## Collaboration & administration: verifiable points

### ChatGPT Business
- Central admin controls and workspace model are explicit in OpenAI’s Business overview and billing/management documentation. [4][7]
- **Data control / training stance:** OpenAI states you **own and control** business data and that it **is not used to train models by default** for Business offerings. [10][11]

### Claude Pro
- Pro is an **individual tier**; team-workspace administration belongs to Anthropic’s team tier (not Pro). Pro-specific admin/security controls for organizations are therefore not the same category as ChatGPT Business. [6]

---

## Independent reviews & user feedback (evidence summary)

Because you asked for separation between anecdotal and systematic evidence, here’s what I can support from retrieved materials:

### Systematic review / platform-style summaries
- A G2-oriented article claims high satisfaction metrics for **Claude** (e.g., “ease of use,” etc.) and includes comparative discussion for ChatGPT vs Claude based on G2 data. However, the retrieved text is not sufficient for me to verify the exact dataset/methodology beyond what’s quoted in that article. Treat these as **directional** and verify against the original G2 pages if you need audit-grade review data. [15]

### Recurring strengths/weaknesses (from retrieved review-style sources; not primary)
- **Claude strengths (commonly cited in review articles):** ease of use and a streamlined experience for writing/research/code workflows. [15]
- **ChatGPT strengths (commonly cited):** richer feature ecosystem and team/admin capabilities on business plans. This is more consistent with official positioning than with review evidence in the retrieved set. [4][7]

**Important limitation:** I did not retrieve multiple independent, credible review platforms specifically comparing *ChatGPT Team/Business* to *Claude Pro* side-by-side with quantified ratings in the current evidence set. So I can’t produce a robust “systematic review” comparison of strengths/weaknesses specific to those two exact plan tiers.

---

## Best-fit scenarios (decision guide)

| User type | More likely fit | Why (based on verified evidence) |
|---|---|---|
| **Individual professional (single user)** | **Claude Pro** *or* OpenAI individual plan | Claude Pro is explicitly an individual plan tier, whereas ChatGPT Business is aimed at managed workspaces. [6][1] |
| **Small team (2–20) needing shared workspace + admin** | **ChatGPT Business (ChatGPT Team)** | Dedicated collaborative workspace, admin console, and workspace-level controls are core to Business. [1][4][7] |
| **Developer** | Tie (depends on coding workflow) | Both ecosystems include coding assistants (Codex in OpenAI Business; Claude Code in Anthropic Pro), but detailed numeric capability comparisons were not fully verifiable from official sources here. [4][5] |
| **Researcher / knowledge work** | **ChatGPT Business** (if team) / **Claude Pro** (if individual) | ChatGPT Business explicitly includes **Deep Research** and knowledge/workspace features. [4] |
| **Organization (compliance/security + centralized management)** | **ChatGPT Business** (and/or Enterprise) | OpenAI provides explicit business data ownership/control and “no training on business data by default” commitments, plus security governance. [10][11] Anthropic’s Pro privacy is consumer-oriented with training choice; for enterprise-grade controls you’d generally use a team/enterprise offering (not Pro). [12][13] |

---

## Limitations and caveats (read this)
1. **Claude Pro pricing and usage limits:** In the retrieved evidence set, I did **not** obtain primary Anthropic pricing/limits pages for Pro, so numeric claims about price/limits are **not fully “officially verified.”** [6]
2. **Claude Pro tool feature list:** Some tools (web/research/cowork/etc.) were not fully verified from official Pro documentation in the retrieved sources.
3. **Plan mismatch:** ChatGPT Business (team) vs Claude Pro (individual) are different product archetypes. A fair like-for-like would compare **ChatGPT Business** vs **Claude Team**, not Claude Pro. I followed your request to compare against Claude Pro, but the interpretation should reflect this mismatch. [1][6]
4. **“Latest as of”**: Evidence was gathered using web search and verified against retrieved sources, and the check date is **2026-11-02**.

---

## Final balanced verdict
- If you truly need a **team workspace** with centralized administration and shared organizational data boundaries, **OpenAI ChatGPT Business (formerly Team)** is the clearer match—this is explicitly its positioning and includes collaboration/workspace constructs plus codex/deep research/work features. [1][4][7][10]
- If you need a **single-user pro subscription** and want Claude’s consumer pro experience, **Claude Pro** is the more natural fit—but based on the retrieved evidence, I can’t fully substantiate every pricing/usage numeric detail from Anthropic official sources here. [6]

**Bottom line:**  
- **Small team/workspace admin needed → choose ChatGPT Business.**  
- **Solo pro user optimizing for personal capacity/Claude’s workflow → choose Claude Pro.**

---

## Sources
1. OpenAI Help Center — **ChatGPT Business release notes** (rename announcement, Aug 29 2025). https://help.openai.com/en/articles/11391654-chatgpt-business-release-notes  
2. OpenAI (OpenAI.com) — **Introducing ChatGPT Team** (includes Aug 29 2025 update: renamed to ChatGPT Business). https://openai.com/index/introducing-chatgpt-team  
3. (Background positioning from the rename/release notes; see sources 1/2)
4. OpenAI Help Center — **ChatGPT Business overview** (features + pricing by seat). https://help.openai.com/en/articles/8792828-chatgpt-business-overview  
5. (Used only indirectly via tool results; no official Pro feature list captured here)
6. Third-party summary for Claude Pro pricing/features (not primary; verify against Anthropic pricing page). https://www.ai-toolbox.co/claude-models/claude-pricing-plans-2026  
7. OpenAI (Help) — **ChatGPT Team / Business billing & seats management** (minimum seats and seat billing details). https://help.openai.com/en/articles/8792536-managing-billing-and-seats-in-chatgpt-business  
8. OpenAI Help Center — **Projects in ChatGPT** (file upload limits: 10 simultaneous, 40 per project on Business). https://help.openai.com/en/articles/10169521-projects-in-chatgpt  
9. Anthropic Platform Docs — **Vision** (image processing capability; developer documentation). https://platform.claude.com/docs/en/build-with-claude/vision  
10. OpenAI — **Enterprise privacy** (data ownership/control; no training by default; limited access). https://openai.com/enterprise-privacy  
11. OpenAI — **Business data privacy, security, and compliance** (no training on organization’s data by default; encryption; trust/security). https://openai.com/business-data  
12. Anthropic News — **Updates to our consumer terms** (training/retention choice; 5-year extension when opted in; 30-day otherwise). https://www.anthropic.com/news/updates-to-our-consumer-terms  
13. Anthropic Privacy Center — **How long do you store my data?** https://privacy.claude.com/en/articles/10023548-how-long-do-you-store-my-data  
14. OpenAI Help Center — **Managing billing and seats in ChatGPT Business** (workspace credits to extend usage beyond included rate limits for Codex/models). https://help.openai.com/en/articles/8792536-managing-billing-and-seats-in-chatgpt-business  
15. G2-oriented article (secondary summarization of user satisfaction). https://learn.g2.com/claude-vs-chatgpt  

If you want, I can redo the comparison with **Claude Team** (instead of Claude Pro) to make the plans more apples-to-apples for collaboration/admin—and I’ll only use Claude Team pricing/limit pages if I can retrieve Anthropic’s official documentation.

In [35]:
# A lightweight manual evaluation table.
# Complete this after reviewing the three generated answers.

evaluation = [
    {"exercise": "1 — Healthcare", "relevant": None, "date_scope_correct": None,
     "sources_traceable": None, "balanced": None, "complete": None, "notes": ""},
    {"exercise": "2 — Startups", "relevant": None, "date_scope_correct": None,
     "sources_traceable": None, "balanced": None, "complete": None, "notes": ""},
    {"exercise": "3 — Product comparison", "relevant": None, "date_scope_correct": None,
     "sources_traceable": None, "balanced": None, "complete": None, "notes": ""},
]

import pandas as pd
evaluation_df = pd.DataFrame(evaluation)
evaluation_df

INFO:numexpr.utils:NumExpr defaulting to 8 threads.


,exercise,relevant,date_scope_correct,sources_traceable,balanced,complete,notes
0,1 — Healthcare,None,None,None,None,None,
1,2 — Startups,None,None,None,None,None,
2,3 — Product comparison,None,None,None,None,None,


## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.


## Bonus Task: Build a Scientific R&D Intelligence Agent
Use case: monitor AI and scientific advances in electrocatalysis and clean energy
A research scientist or R&D team needs to monitor new literature, technology announcements, and funding or partnership developments related to:
- electrocatalysis and oxygen/hydrogen reactions;
- water electrolysis and fuel cells;
- platinum-group-metal reduction and non-precious catalysts;
- nanostructured electrodes and catalyst layers;
- impedance spectroscopy and electrochemical characterization;
- AI/ML for materials discovery and process optimization.
The agent should find relevant work, summarize the scientific contribution, identify evidence and limitations, and suggest experiments or data-science opportunities. It must not fabricate DOI numbers, performance values, or publication details.

In [36]:
prompt_bonus = """
Act as a scientific R&D intelligence analyst supporting a multidisciplinary
team working on electrocatalysis, electrochemical energy conversion, and
AI-assisted materials discovery.

Search the public web for important research and technology developments
from the last 12 months relative to the date of execution. Prioritize:
- oxygen reduction/evolution and hydrogen evolution reactions;
- alkaline water electrolysis and fuel-cell technologies;
- Pt-group-metal utilization, non-precious catalysts, and catalyst durability;
- nanowire/network electrodes, porous catalyst layers, and advanced supports;
- electrochemical impedance spectroscopy and operando characterization;
- machine learning for catalyst discovery, materials optimization, and
  electrochemical data analysis.

For each of 5–8 highly relevant findings, report:
1. Title, authors/organization, publication or announcement date.
2. Journal/conference or organization, and DOI or direct URL if verified.
3. Scientific question and the approach used.
4. Main quantitative findings, including units and test conditions when available.
5. What is genuinely new compared with prior approaches.
6. Limitations, uncertainties, and whether the evidence is peer reviewed.
7. Potential relevance to electrocatalyst R&D and a practical follow-up experiment
   or data-science analysis.

Then provide:
- A ranked list of 3 research opportunities.
- A proposed small experimental/ML workflow, including candidate input features,
  target variables, validation strategy, and risks of data leakage.
- A short list of useful datasets or open repositories, with direct links.

Use primary scientific papers, institutional repositories, and official project
pages wherever possible. Never invent a DOI, numerical result, or citation.
If full-text evidence is unavailable, clearly label what is based only on
the abstract or announcement.

Output clean Markdown with an executive summary, findings table, deeper analysis,
proposed workflow, and numbered sources. State the search date.
"""

if research_agent:
    response_bonus = run_research(research_agent, prompt_bonus)
    display_run("Bonus — Scientific R&D Intelligence", response_bonus)
else:
    print("Configure API keys and initialize the agent to run the bonus task.")

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


## Bonus — Scientific R&D Intelligence
Tools called: ['web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search', 'web_search']
Model calls: 7
Tokens (input/output/total): 55887 4853 60740


## Executive summary (search date: 2026-11-02)
I searched the public web for **major electrocatalysis / electrochemical energy conversion / AI-assisted materials discovery developments from the last ~12 months** relative to today, prioritizing **ORR/OER/HER**, **alkaline water electrolysis & fuel cells**, **Pt-group utilization**, **durability**, **nanowire/porous electrodes & supports**, **EIS & operando**, and **ML for catalyst discovery/optimization/data analysis**.

Below are **7 highly relevant findings** (some are “hard-news” research papers; others are official/organization-level publications where DOI/quant results are less consistently exposed on the landing pages). Where I could not verify full-text quantitative details or DOI directly from the page, I explicitly label the evidence as **announcement/abstract-level**.

---

## Findings (5–8 highly relevant developments)

### Table: prioritized “last 12 months” electrocatalysis & ML-relevant findings

| # | Title (authors/org) | Date | Venue + DOI/URL | Scientific question & approach | Main quantitative findings (with conditions) | What’s genuinely new vs prior work | Limitations / evidence quality | Practical relevance & follow-up experiment/analysis |
|---:|---|---|---|---|---|---|---|---|
| 1 | **“Sandwiching intermetallic Pt3Fe and ionomer with porous N-doped carbon layers for oxygen reduction reaction”** (X. Cao et al.) | **2025-03-23** (article page) | **Nature Communications** — DOI shown on journal page: **10.1038/s41467-025-58116-7** | How to **protect Pt-group active sites** from **ionomer / poisoning effects** while improving ORR performance. Uses **core/shell intermetallic Pt3Fe + ionomer sandwich** with **atomically-thin porous N-doped carbon** for site protection and transport. | *Not fully extractable from search snippet alone.* Journal page states conceptually key effects (“prevent poisoning”, “increase active sites”), but I did not retrieve the full quantitative performance (mass activity / power density / overpotential) from the provided tool content. | “Intermetallic core + ionomer + porous N-doped carbon sandwich” to decouple **active-site chemistry** from **ionomer-related degradation/poisoning**, while maintaining ORR transport/three-phase boundaries. | Needs **full-text** to extract exact quantitative metrics; evidence is based on journal landing page content only (**not full peer-reviewed metrics captured here**). | **Relevance:** direct strategy for **higher Pt utilization** and **durability** in PEMFC cathodes. **Follow-up:** perform accelerated stress tests (potential cycling) + quantify ECSA loss and ionomer-related deactivation; pair with **in-operando XAS** (Pt oxidation state/coordination) to validate the “site protection” mechanism. |
| 2 | **“Insights into catalyst degradation during alkaline water electrolysis under variable operation”** (R. A. Marquez et al.) | **2025** (article citation on RSC page; exact month not captured) | **Energy & Environmental Science** — URL shown by search: https://pubs.rsc.org/ee/article/18/14/7170/891343/Insights-into-catalyst-degradation-during-alkaline | What **degradation pathways** dominate in alkaline HER/OER catalysts under **realistic variable operation** (dynamic loads). Approach combines electrochemical testing with degradation/characterization to link operation mode to catalyst changes. | *Quantitative metrics not extractable from the search snippet text provided.* The snippet indicates “severe degradation” and discusses Ni/Fe/Co catalysts, but no current densities / mV shifts / timescales were captured. | Focus on **variable operation** (not just steady-state galvanostatic holds) and how it drives **catalyst degradation**—a practical gap for AWE/LAE lifetimes under load-following. | Quantitative details require full-text; evidence here is **abstract/landing-page-level**. Still likely peer-reviewed (E&ES). | **Relevance:** informs durability models and test protocols for alkaline catalysts. **Follow-up:** replicate with **realistic dynamic current profiles**, then use **EIS/DRT** during transients to separate ohmic/charge-transfer vs diffusion contributions as degradation progresses. |
| 3 | **“Operando unveiled the activity origin via preferential …”** (Z. P. Wu et al.) | **2025** (Science DOI indicates 2025; exact day not extracted) | **Science Advances** — DOI **10.1126/sciadv.adu5370** | Where does the **activity originate** in an OER catalyst system? Uses **operando** methods to distinguish mechanistic origins (e.g., active phase formation vs bulk behavior). | *Quantitative results not retrievable from the snippet content.* Science Advances landing content only provides topic-level description. | “Activity origin” resolved with **preferential/operando** evidence rather than ex situ post-mortem alone—moving toward **causal mechanistic assignment**. | Need full text for quantitative claims and methods detail; evidence here is **announcement-level**. Peer-reviewed likely (Science Advances). | **Relevance:** helps validate mechanistic descriptors used in electrocatalyst screening. **Follow-up:** use their mechanistic “active phase” signature as ML labels; train a model to predict activity from operando spectral features (XAS/XPS peak ratios) vs electrochemical metrics. |
| 4 | **“Operando X-ray absorption spectroscopy unveils light-driven redox dynamics at the semiconductor/cocatalyst interface”** (R. Mazzaro et al.) | **Not fully confirmed from snippet** (Science Advances page indicates “sciadv.adx8089”) | **Science Advances** — URL: https://www.science.org/doi/10.1126/sciadv.adx8089 | How oxidation-state dynamics change under **operando illumination** at semiconductor/cocatalyst interfaces. Combines **operando XAS** with **fixed-energy X-ray absorption voltammetry (FEXRAV)**. | *No quantitative values captured from snippet.* | Demonstrates **element-selective, state-resolved operando probing** strategy combining XAS modes, which can be ported conceptually to electrocatalysis interfaces under potential/field/flow perturbations. | This is more “photoelectrocatalysis adjacent” than pure HER/OER, so direct relevance to AWE may be limited, but methods are transferable. Evidence quality: peer-reviewed (Science Advances) but quant details not extracted here. | **Relevance:** methodological template for operando mechanistic studies. **Follow-up:** adapt “fixed-energy absorption voltammetry” concept to potential ramps in AWE (Ni/Fe/Cox) to quantify oxidation-state kinetics during dynamic load. |
| 5 | **“Operando Impedance Spectroscopy Informed Dynamic …”** (B. Tomc et al.) | **2025** (from tool snippet) | **ACS** — URL shown: https://pubs.acs.org/aeclc7/article/1/12/2823/3713844/Operando-Impedance-Spectroscopy-Informed-Dynamic | How to use **operando EIS** to infer **dynamic processes** in electrocatalysis (likely HER context from the page context snippet). | *No numeric parameters captured from snippet.* | Direct coupling of **operando EIS** with **dynamic operation/overpotential analysis** to connect measured relaxation features to catalyst/electrode states. | Full text needed for quantitative extraction; evidence is **announcement/landing-page-level**. | **Relevance:** improves how we extract mechanistic fingerprints from EIS. **Follow-up:** run EIS during current-step experiments; use **distribution of relaxation times (DRT)** and map relaxation peaks to degradation mechanisms (e.g., charge transfer resistance vs double-layer vs mass transport). |
| 6 | **“Advanced characterization of alkaline water electrolysis through electrochemical impedance spectroscopy and polarization curves”** | *2024 DOI shown in portal; may fall near last-12-month window depending on publication date vs access date—needs verification.* | **Journal of Electroanalytical Chemistry** — DOI: **10.1016/j.jelechem.2024.118709** (portal shows) | How to extract more value from EIS beyond simple IR correction in alkaline water electrolysis. Combines EIS with polarization to interpret kinetics/resistances/capacitance contributions. | *No quantitative results captured from snippet.* | Shifts EIS use from “IR correction” to **kinetic and capacitive/charge-transfer diagnostics** tailored to AWE. | The portal indicates peer-reviewed article (“Academic › peer-review”), but I could not extract numbers. Evidence: portal metadata + abstract-level. | **Relevance:** directly supports building better **EIS-based descriptors** for catalyst and electrode layer evaluation. **Follow-up:** implement a standardized EIS pipeline to compute physically interpretable parameters (Rct, Cdl/CCE proxies) and validate against independent measurements (H2 production rates). |
| 7 | **“Pourbaix machine learning framework identifies acidic water oxidation catalysts exhibiting suppressed ruthenium dissolution”** (J. Abed et al.; Sargent group; APS highlight references JACS 2024 paper) | **2024** (highlight is 2024-11-08; paper year appears 2024) | JACS page not retrieved in tool results; APS highlight URL: https://www.aps.anl.gov/APS-Science-Highlight/2024-11-08/using-machine-learning-to-find-better-electrochemical-catalysts | ML for catalyst selection using a Pourbaix-informed representation; then experimental validation. Targeting durability via **suppressed Ru dissolution**. | *No quantitative dissolution suppression captured from snippet.* | ML framework that couples electrochemical thermodynamics (Pourbaix) to candidate selection for improved **durability** rather than only activity. | Not strictly “last 12 months” from today if the underlying paper is 2024; I’m including it because it’s highly relevant to your “ML + operando durability” priority and because the APS highlight provides a verified organizational context. Evidence is **not full-text** for numeric metrics. | **Relevance:** provides a template for **electrochemical-thermodynamics-aware ML**. **Follow-up:** develop an ML model for AWE catalyst dissolution/degradation using Pourbaix-like stability maps and train on operando dissolution proxies (e.g., ICP-MS time series). |

> **Note on “last 12 months” rigor:** For a few entries, the web snippets returned DOIs/years but not the exact publication day or the last-12-month boundary with certainty. Where that matters, the limitations column flags what would require verification from the full journal page/PDF. If you want, I can rerun the search with stricter filters for exact publication dates once you confirm your execution date window definition (e.g., “from 2025-11-02 to 2026-11-02”).

---

## Deeper analysis: cross-cutting themes for electrocatalyst R&D & AI/ML

### 1) Durability is increasingly treated as a *mechanistic* and *test-protocol* problem
- The alkaline degradation work under **variable operation** highlights that “lab steady-state” can miss failure modes that appear in realistic load-following [2].
- Takeaway: **model inputs should include operational history** (current density trajectory, rest/relaxation steps, bubble coverage proxies) rather than only static polarization curves.

### 2) Pt-group-metal utilization gains are coming from **architecture + interface engineering**
- The Pt3Fe/ionomer/porous N-carbon sandwich strategy is representative: it aims to increase accessible active-site fraction and reduce poisoning while preserving ORR transport [1].
- Takeaway: prioritize **interfaces** (ionomer–catalyst, pore–ionomer–water pathways) as first-class variables for both experimental design and ML features.

### 3) EIS and operando methods are moving from “diagnostics” to “dynamic fingerprints”
- Operando impedance informed by dynamics [5] and advanced EIS use beyond IR correction for AWE [6] support the trend: use **frequency-resolved**, **time-dependent** information to extract kinetic vs mass-transport signatures.
- Takeaway: for ML, EIS should be treated as a **spectral time series** or as **DRT peak parameters** with uncertainty estimates, not as a single scalar Rct.

### 4) ML frameworks are increasingly coupling to electrochemical stability constraints
- Pourbaix-aware ML for catalyst selection/durability is a strong pattern [7].
- Takeaway: incorporate **thermodynamic stability features** (phase/pourbaix windows, dissolution likelihood surrogates) to reduce “activity-only” failure modes.

---

## Ranked list of 3 research opportunities

1. **Build an “EIS-to-degradation” model for alkaline electrolysis**
   - Combine operando EIS (and/or DRT peaks) with degradation endpoints (ECSA loss, dissolution species quantified by ICP-MS, metal oxidation-state shifts).
   - Goal: predict lifetime degradation rates under variable current trajectories.

2. **Interface-engineering descriptors for ORR durability in Pt-group catalysts**
   - Use the Pt3Fe sandwich design concept to generate a dataset mapping **interface structure parameters** (porous N-carbon thickness, pore connectivity, ionomer distribution metrics) → ORR activity + durability under stress.
   - Goal: ML that optimizes for **activity retained after AST**, not just as-prepared performance.

3. **Thermodynamics-constrained ML screening for dissolution-resistant catalysts (AWE + ORR)**
   - Extend Pourbaix-ML concepts to alkaline HER/OER catalysts where dissolution and phase changes are dominant.
   - Goal: rank candidates by **stability margin** under specific operating windows rather than only overpotential.

---

## Proposed small experimental/ML workflow (with leakage risk controls)

### Objective
Predict catalyst **durability** (e.g., activity retention or dissolution rate) from **electrochemical + operando features**, enabling candidate ranking and design iterations.

### Dataset structure
For each catalyst/electrode condition *i*:
- Inputs: operando/pseudo-operando measurements during designed current/voltage profiles.
- Targets: durability metrics from post-test characterization.

### Candidate input features
**Electrochemical**
- Polarization curve-derived features: E@j (potential at current density), Tafel slope, hysteresis area
- EIS/DRT features: RΩ, Rct, CPE parameters, DRT peak centers & areas, integrated impedance in frequency bands
- Dynamic features: response time constants from EIS during current steps, transient overpotentials

**Materials / structure (from characterization)**
- Metal loading, particle size distribution (from TEM), porosity metrics (BET, mercury intrusion)
- Support properties (surface area, conductivity proxy)
- Composition/phase fractions (XRD) and oxidation states (XPS/XAS where available)
- Interface descriptors: ionomer coverage proxy, pore size distribution relevant to water transport

**Operational profile encoding**
- Current density trajectory statistics: mean, max, ramp rate, duty cycle, number of cycles, rest times

### Target variables (choose 1–2 to start)
- Primary: **activity retention** after accelerated stress test (AST) (e.g., % ECSA or overpotential shift)
- Secondary: **dissolution rate** proxy (if ICP-MS available) or oxidation-state evolution rate from operando XAS

### Modeling approach
- Start with **regularized regression / gradient boosting** (handles small data well).
- Move to **Gaussian Processes** or Bayesian regression for uncertainty quantification.
- For EIS: either (a) DRT-parameter models, or (b) frequency-band features; avoid training directly on raw spectra if data are scarce.

### Validation strategy (anti-leakage)
- **Group-based split**: split by *catalyst formulation batch* (or by synthesis batch) so that near-duplicates don’t appear in both train and test.
- **Time split** only if measurements are sequentially generated; otherwise group split is safer.
- Hyperparameter tuning: nested cross-validation within training groups only.
- Ensure that **AST-derived targets** are not used to compute any input features that would be available only after AST.

### Data leakage risks to explicitly guard against
- Reusing the same sample’s **pre/post** features in both train and test (group split fixes).
- Using post-mortem-derived features (e.g., corrosion morphology after AST) as inputs if your workflow intends *pre-optimization*.
- “Metadata leakage”: e.g., encoding “AST run ID” or “lab instrument ID” correlated with outcomes.

---

## Useful datasets / open repositories (starting points)
I did not find definitive, directly relevant **open alkaline/AWE operando EIS datasets** in the tool results provided. However, these are good starting references (you may want to re-check each for licensing and completeness):

- **Mendeley dataset listings / EIS data-driven DRT items** (leads to potentially relevant datasets, but requires vetting): “data-driven analysis of electrochemical impedance … DRT through Loewner framework” [dataset listing page returned in tool results]  
- **Institutional repository examples** (for EIS-heavy thesis datasets): Eindhoven/TU repositories often host EIS+polarization raw data (example metadata surfaced via TU Eindhoven portal for AWE EIS work) [6]

If you want, tell me whether you prefer **public raw EIS spectra**, **DRT-processed values**, or **fulloperando time-series**—and I’ll do a second targeted search for actual downloadable datasets.

---

## Conclusion
Across the last ~year, the strongest actionable signals are:
1) **Durability under realistic dynamic operation** is becoming a central scientific question [2].  
2) Pt-group gains increasingly come from **interface and architecture engineering**, not just “more Pt” [1].  
3) **Operando EIS/DRT** and dynamic characterization are enabling mechanistic “fingerprints” for both activity and degradation [5,6].  
4) ML is shifting toward **thermodynamics/stability-aware** frameworks to reduce “activity-only” screening pitfalls [7].

---

## Sources
1. Nature Communications — *Sandwiching intermetallic Pt3Fe and ionomer with porous N-doped carbon layers for oxygen reduction reaction* (X. Cao et al.). DOI/URL: https://www.nature.com/articles/s41467-025-58116-7  
2. RSC Energy & Environmental Science — *Insights into catalyst degradation during alkaline water electrolysis under variable operation* (R. A. Marquez et al.). URL: https://pubs.rsc.org/ee/article/18/14/7170/891343/Insights-into-catalyst-degradation-during-alkaline  
3. Science Advances — *Operando unveiling the activity origin via preferential ...* (Z. P. Wu et al.). DOI/URL: https://www.science.org/doi/10.1126/sciadv.adu5370  
4. Science Advances — *Operando X-ray absorption spectroscopy unveils light-driven redox dynamics at the semiconductor/cocatalyst interface* (R. Mazzaro et al.). DOI/URL: https://www.science.org/doi/10.1126/sciadv.adx8089  
5. ACS (ACS author/page) — *Operando Impedance Spectroscopy Informed Dynamic ...* (B. Tomc et al.). URL: https://pubs.acs.org/aeclc7/article/1/12/2823/3713844/Operando-Impedance-Spectroscopy-Informed-Dynamic  
6. TU Eindhoven research portal — *Advanced characterization of alkaline water electrolysis through electrochemical impedance spectroscopy and polarization curves* (metadata includes DOI). DOI/URL: https://research.tue.nl/en/publications/advanced-characterization-of-alkaline-water-electrolysis-through-  (DOI: 10.1016/j.jelechem.2024.118709 shown in the portal)  
7. Argonne/APS science highlight — *Using Machine Learning to Find Better Electrochemical Catalysts* (references the JACS Pourbaix-ML work). URL: https://www.aps.anl.gov/APS-Science-Highlight/2024-11-08/using-machine-learning-to-find-better-electrochemical-catalysts  

--- 

If you confirm (a) your exact “last 12 months” cutoff date and (b) whether you need **only primary research papers** (excluding reviews/highlights), I can redo the search to tighten the date window and ensure **each entry has fully verified DOIs + extracted numeric metrics** from the full text wherever accessible.

In [53]:
# Final reflection — complete in your own words.

conclusion = """
1. The most important lesson about designing research prompts: ...
2. The most important lesson about source quality: ...
3. A limitation I observed in the agent: ...
4. One improvement I would make before real-world deployment: ...
"""
print(conclusion)


1. The most important lesson about designing research prompts: ...
2. The most important lesson about source quality: ...
3. A limitation I observed in the agent: ...
4. One improvement I would make before real-world deployment: ...



In [52]:
prompt_bonus = """
Act as a scientific R&D intelligence analyst supporting a multidisciplinary
team working on electrocatalysis, electrochemical energy conversion, and
AI-assisted materials discovery.

Search the public web for important research and technology developments
from the last 12 months relative to the date of execution. Prioritize:
- Machine learning Force Fields 
- Material prediction/discovery using machine learning and quantum mechanics 
- Density functional theory 
- Electrochemical theory 
- Electrochemistry related to materials discovery, machine learning and quantum mechanics
- Ab initio  molecular dynamics 
- machine learning for catalyst discovery, materials optimization, and
  electrochemical data analysis.

For each of 5–8 highly relevant findings, report:
1. Title, authors/organization, publication or announcement date.
2. Journal/conference or organization, and DOI or direct URL if verified.
3. Scientific question and the approach used.
4. Main quantitative findings, including units and test conditions when available.
5. What is genuinely new compared with prior approaches.
6. Limitations, uncertainties, and whether the evidence is peer reviewed.
7. Potential relevance to electrocatalyst R&D and a practical follow-up experiment
   or data-science analysis.

Then provide:
- A ranked list of 3 research opportunities.
- A proposed small experimental/ML workflow, including candidate input features,
  target variables, validation strategy, and risks of data leakage.
- A short list of useful datasets or open repositories, with direct links.

Use primary scientific papers, institutional repositories, and official project
pages wherever possible. Never invent a DOI, numerical result, or citation.
If full-text evidence is unavailable, clearly label what is based only on
the abstract or announcement.

Output clean Markdown with an executive summary, findings table, deeper analysis,
proposed workflow, and numbered sources. State the search date.
"""

if research_agent:
    response_bonus = run_research(research_agent, prompt_bonus)
    display_run("Bonus — Scientific R&D Intelligence", response_bonus)
else:
    print("Configure API keys and initialize the agent to run the bonus task.")


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


## Bonus — Scientific R&D Intelligence
Tools called: ['web_search', 'web_search', 'web_search', 'web_search', 'web_search']
Model calls: 6
Tokens (input/output/total): 12262 2403 14665


## Executive summary (search date: 2026-11-02)
Below are **7 highly relevant web-verified findings from roughly the last 12 months** focused on ML force fields / quantum-informed discovery / electrochemical theory & ML for catalyst discovery. Several are **project/report or preprint-level** rather than fully peer-reviewed papers; I flag those cases explicitly.  

## Findings table

| # | Title (authors/organization) | Date | Evidence | Main quantitative result(s) (units, conditions if available) | “New vs prior” | Relevance + practical follow-up |
|---:|---|---|---|---|---|---|
| 1 | **Autonomous Flow Electrochemistry for Accelerated Catalyst Discovery** (Vasisht et al., PNNL) | 2025-10-16 | Official report page [https://www.pnnl.gov/publications/autonomous-flow-electrochemistry-accelerated-catalyst-discovery] | **No numeric performance values shown in the web snippet**; describes closed-loop autonomy across three reactions (formate oxidation, cyclohexanol oxidation, hydroquinone oxidation) | Emphasizes an **autonomous ACE loop** connecting potentials → spectral analysis → decision/optimization | Follow-up: build an ML surrogate for **ACE decision policies** using features derived from voltammograms/spectra and propose new compositions via BO; validate with time-split + reaction-split |
| 2 | **Ensemble Learning of Machine Learning Force Fields** (arXiv:2403.17507; authors not captured in snippet) | (Preprint; version shown 2403.17507v2) | arXiv preprint page [https://arxiv.org/html/2403.17507v2] | Snippet shows a table with **metrics (e.g., “forces↓”, “h(r)↓”, “stability↑”)** but **conditions/materials/authors and exact numeric values are incomplete in the snippet** | Positions **ensembles** to improve ML force-field robustness vs single-model NNPs | Follow-up: test ensemble uncertainty as an **active-learning acquisition function** for electrocatalyst slab/coverage sampling |
| 3 | **Atoms to Processes: The Role of Artificial Intelligence and Machine Learning in Chemical Engineering** (arXiv:2610.02014; includes electrocatalysis references) | 2026-10 (preprint id 2610…) | arXiv PDF/html [https://arxiv.org/pdf/2610.02014], [https://arxiv.org/html/2610.02014v1] | **No specific new quantitative electrocatalysis result extracted** from snippet beyond mentions of robotic/active-learning loops | Synthesis paper—useful for mapping approaches (Bayesian optimization, robotics, active learning) | Follow-up: convert the review’s taxonomy into a **feature/label spec** for electrochemical data-model training |
| 4 | **DFT-FE Publications: Learning local/semi-local functionals etc.** (University of Michigan “DFT-FE” group publications list) | 2025 (multiple items) | Official publications list [https://sites.google.com/umich.edu/dftfe/publications] | Snippet lists specific 2025 works (e.g., “Science Advances 11, eady8962 (2025)” for exact exchange-correlation learning), but **no electrocatalysis-specific numeric results captured** | Targets **data-driven exchange-correlation potentials/functionals**, i.e., quantum-level surrogate learning | Follow-up: evaluate whether learned functionals reduce DFT cost for **electrochemical adsorption energetics** and feed those to ML reaction/kinetic models |
| 5 | **On the importance of catalyst-adsorbate geometric relative information when predicting relaxed energy** (AI4Mat 2026 accepted work list) | 2026 (accepted-work page) | AI4Mat accepted-work page [https://sites.google.com/view/ai4mat/ai4mat-2023/accepted-work] | **No quantitative results captured** in snippet | Introduces a **geometric relational representation** direction for property prediction | Follow-up: implement relative-geometry encodings for **electrocatalyst adsorption + NEB intermediates** prediction |
| 6 | **Development of Universal Neural Network for Materials Discovery** (Preferred Networks blog post) | (No date captured in snippet) | Preferred Networks page [https://www.preferred.jp/en/blog/tech/development-of-universal-neural-network-for-materials-discovery] | **No numeric values captured** | Marketing/overview for “universal” generative/potential approaches | Follow-up: treat as a hypothesis generator for **constraints-aware generation** of electrocatalyst candidates |
| 7 | **DFT and ML Applications in Chemistry and Materials Science for Sustainable Energy Research** (LIDSEN special-issue page) | 2026 (open access publisher page) | LIDSEN page [https://www.lidsen.com/journals/rpse/rpse-special-issues/dft-machine-learning-applications] | **No new quantitative results**; meta-level list | Curated scope rather than a specific method advance | Follow-up: mine the special issue for **exact papers/DOIs** relevant to electrocatalysis ML + DFT |

> **Important limitation:** Several web snippets do not expose DOIs, exact authors, full abstracts, or the numerical values needed for rigorous “quantitative findings.” Where that happens, I’m reporting only what is verifiably shown on the cited pages (no invented DOIs/results).

## Deeper analysis (what seems most “new” for electrocatalysis R&D)
1) **Closed-loop autonomy in electrochemistry** (PNNL ACE report) appears as the most directly transferable advance: it operationalizes optimization by linking **electrochemical execution → analysis → decision** within an autonomous loop. [https://www.pnnl.gov/publications/autonomous-flow-electrochemistry-accelerated-catalyst-discovery]  
2) **Force-field uncertainty + ensembles** (ML force-field ensemble learning) aligns with the need to choose new catalyst surfaces/coverages/adsorbate configurations for DFT/MD resampling. [https://arxiv.org/html/2403.17507v2]  
3) **Quantum-informed surrogate learning (DFT functionals/potentials)** (DFT-FE listings) targets the bottleneck of high-cost electronic-structure calls, potentially enabling faster electrochemical adsorption/energetics screening. [https://sites.google.com/umich.edu/dftfe/publications]  

## Ranked list of 3 research opportunities
1) **Active-learning + ensemble ML force-field/DNN potentials for adsorption/coverage exploration** on catalyst slabs, using **uncertainty** to drive new DFT points (aim: better coverage dependence with fewer expensive calculations). [https://arxiv.org/html/2403.17507v2]  
2) **Autonomous electrochemical catalyst discovery loop**: reproduce ACE-style closed-loop optimization for a specific reaction (e.g., formate oxidation) while training an ML model on voltammetry/spectra to improve proposal quality beyond vanilla BO. [https://www.pnnl.gov/publications/autonomous-flow-electrochemistry-accelerated-catalyst-discovery]  
3) **Learned-DFT surrogate for electrochemical energetics**: benchmark whether learned exchange-correlation models reduce error/cost for adsorption energies and downstream kinetic models. [https://sites.google.com/umich.edu/dftfe/publications]  

## Proposed small experimental/ML workflow (with leakage controls)
- **Inputs (features):** catalyst composition encoding (elements + stoichiometry), surface facet/structure descriptors, adsorbate identity, coverage %, prior-step potential, and electrochemical descriptors from raw/processed data (peak positions, onset potential, integrated currents, spectral features if available).  
- **Targets (labels):** (i) adsorption/ΔG (from DFT or experimentally inferred), (ii) onset potential / activity metric (e.g., current density at fixed potential), and/or (iii) kinetic surrogate parameters if you fit microkinetics.  
- **Modeling:** start with uncertainty-aware ensembles (for active learning) plus a relational geometry encoder for relaxed-energy prediction. [https://arxiv.org/html/2403.17507v2]  
- **Validation:** reaction-split and composition-split (hold out entire catalysts or entire reaction classes) + strict time-split if using sequential experimental logs (prevents “training on the future”).  
- **Leakage risks to explicitly prevent:** duplicate electrocatalyst recipes across folds; normalization fitted on full dataset; time leakage in Bayesian/ACE-style logs where early measurements inform later splits implicitly.

## Useful datasets / repositories (from the sources found)
- **Not found with direct dataset links in the retrieved snippets**; the web results here were primarily publication pages. Use the identified sources to pivot to their linked code/data repositories when available (e.g., arXiv projects typically link to GitHub; ACE reports may link supplementary materials). [https://www.pnnl.gov/publications/autonomous-flow-electrochemistry-accelerated-catalyst-discovery], [https://arxiv.org/html/2403.17507v2]

---

### Numbered sources
1. https://www.pnnl.gov/publications/autonomous-flow-electrochemistry-accelerated-catalyst-discovery  
2. https://arxiv.org/html/2403.17507v2  
3. https://arxiv.org/pdf/2610.02014  
4. https://arxiv.org/html/2610.02014v1  
5. https://sites.google.com/umich.edu/dftfe/publications  
6. https://sites.google.com/view/ai4mat/ai4mat-2023/accepted-work  
7. https://www.preferred.jp/en/blog/tech/development-of-universal-neural-network-for-materials-discovery  
8. https://www.lidsen.com/journals/rpse/rpse-special-issues/dft-machine-learning-applications  

If you want, I can rerun the web search with tighter queries **specifically including DOIs and electrochemistry keywords** (e.g., “electrocatalysis machine learning active learning voltammetry DOI 2026” and “neural network potential catalysis electrochemistry DOI”), so the 5–8 items include verifiable quantitative results and DOIs.

## Extra

In [38]:
import os
import re
import ast
import operator
import logging
import warnings
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

# Resolve the CSV whether the notebook is run from your-code/ or the project root.
CANDIDATE_PATHS = [
    Path("../data/bank_documents.csv"),
    Path("data/bank_documents.csv"),
    Path("/mnt/data/LAB3_completed/data/bank_documents.csv"),
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find data/bank_documents.csv. Check the project folder layout.")

# Load .env from the current directory or the notebook folder.
load_dotenv()
OPENAI_AVAILABLE = bool(os.getenv("OPENAI_API_KEY"))
TAVILY_AVAILABLE = bool(os.getenv("TAVILY_API_KEY"))

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano")
# Illustrative prices from the original exercise; verify current provider pricing before budgeting.
PRICE_INPUT_PER_MILLION = 0.20
PRICE_OUTPUT_PER_MILLION = 1.25

documents = pd.read_csv(DATA_PATH)
required_columns = {"id", "title", "text"}
assert required_columns.issubset(documents.columns), f"Missing columns: {required_columns - set(documents.columns)}"
assert len(documents) == 10, f"Expected 10 course documents, found {len(documents)}"

print(f"Data loaded: {documents.shape[0]} documents × {documents.shape[1]} columns")
print(f"OpenAI key present: {OPENAI_AVAILABLE} | Tavily key present: {TAVILY_AVAILABLE}")
print(f"Model setting: {MODEL}")
display(documents)

Data loaded: 10 documents × 3 columns
OpenAI key present: True | Tavily key present: True
Model setting: gpt-5.4-nano


,id,title,text
0,premium-1,Premium plan,The Premium plan costs 9.99 euros per month. I...
1,customers-1,Customer base,"In September 2026 the bank had 182,000 active ..."
2,complaints-1,"Complaints, third quarter of 2026","The bank received 1,240 complaints between Jul..."
3,app-1,Mobile app,In September 2026 the mobile app was rated 4.3...
4,roadmap-1,Product roadmap 2027,Planned for 2027: an AI assistant called Nova ...
5,fx-1,Foreign currency,Card payments in a foreign currency use the Ma...
6,cards-2,Card delivery,New cards arrive within 7 working days. If a c...
7,cards-3,Blocked PIN,After three wrong PIN attempts the card is blo...
8,transfers-1,Transfer limits,"Customers can send up to 10,000 euros per day ..."
9,ai-1,AI assistant policy,The support chatbot must start every conversat...


In [39]:
from pathlib import Path
import os, re, ast, operator, json, math, warnings
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 30)

# Resolve data relative to this notebook, whether run from the project root or your-code/.
CANDIDATE_DATA_PATHS = [
    Path("../data/bank_documents.csv"),
    Path("data/bank_documents.csv"),
    Path("../../data/bank_documents.csv"),
]
DATA_PATH = next((p for p in CANDIDATE_DATA_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find data/bank_documents.csv. Keep the data folder beside your-code/.")
documents = pd.read_csv(DATA_PATH).fillna("")
required_columns = {"id", "title", "text"}
assert required_columns.issubset(documents.columns), f"Missing columns: {required_columns - set(documents.columns)}"
assert documents["id"].is_unique, "Document IDs must be unique."

# Load .env from the notebook folder and current working directory (without printing secrets).
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
MODEL = os.getenv("OPENAI_MODEL", "openai:gpt-5.4-nano")

# Illustrative defaults only; update from the provider's current pricing page.
INPUT_PRICE_PER_MILLION = float(os.getenv("INPUT_PRICE_PER_MILLION", "0.20"))
OUTPUT_PRICE_PER_MILLION = float(os.getenv("OUTPUT_PRICE_PER_MILLION", "1.25"))

print(f"Loaded {len(documents)} internal documents.")
print("OpenAI credentials configured:", bool(OPENAI_API_KEY))
print("Tavily credentials configured:", bool(TAVILY_API_KEY))
display(documents)

Loaded 10 internal documents.
OpenAI credentials configured: True
Tavily credentials configured: True


,id,title,text
0,premium-1,Premium plan,"The Premium plan costs 9.99 euros per month. It includes card payments in foreign currencies with no fee, travel insurance, and two free..."
1,customers-1,Customer base,"In September 2026 the bank had 182,000 active customers, of whom 14% were on the Premium plan. 61% of customers are aged 25 to 44."
2,complaints-1,"Complaints, third quarter of 2026","The bank received 1,240 complaints between July and September 2026: 38% about card delivery, 22% about blocked cards, 15% about transfer..."
3,app-1,Mobile app,In September 2026 the mobile app was rated 4.3 out of 5 in the app stores. The features customers request most are budgeting tools and i...
4,roadmap-1,Product roadmap 2027,"Planned for 2027: an AI assistant called Nova in the app in the first quarter, budgeting tools in the second quarter, and a joint accoun..."
5,fx-1,Foreign currency,Card payments in a foreign currency use the Mastercard exchange rate plus a 1.5% fee. The fee is waived for Premium customers.
6,cards-2,Card delivery,"New cards arrive within 7 working days. If a card has not arrived after 10 working days, the customer can order a free replacement in th..."
7,cards-3,Blocked PIN,"After three wrong PIN attempts the card is blocked. The customer can unblock it in the app under Card > PIN > Unblock, or at any of the ..."
8,transfers-1,Transfer limits,"Customers can send up to 10,000 euros per day by SEPA transfer. Instant transfers are limited to 2,000 euros each."
9,ai-1,AI assistant policy,"The support chatbot must start every conversation with: 'You are chatting with Nova, the bank's AI assistant.' Staff must never paste cu..."


## A calculator tool with a restricted expression grammar
The section of the lab asks for a calculator that accepts only digits, whitespace, decimal points, parentheses, and arithmetic operators. The original exercise suggests eval() after a character check. A safer implementation parses the expression with Python's AST and permits only numeric constants and a small set of arithmetic nodes. This prevents function calls, attribute access, imports, and arbitrary Python execution.

We implement a local function first so it can be tested even when LangChain is not installed. When LangChain is available, we wrap it with @tool.

In [40]:
_ALLOWED_BINARY_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
}
_ALLOWED_UNARY_OPERATORS = {ast.UAdd: operator.pos, ast.USub: operator.neg}
_ALLOWED_CHARS = re.compile(r"^[0-9eE+*/().\-\s]+$")

def _safe_eval_node(node):
    if isinstance(node, ast.Expression):
        return _safe_eval_node(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BINARY_OPERATORS:
        left, right = _safe_eval_node(node.left), _safe_eval_node(node.right)
        # Avoid pathological exponentiation / enormous results.
        if isinstance(node.op, ast.Pow) and abs(right) > 10:
            raise ValueError("Exponent is too large.")
        result = _ALLOWED_BINARY_OPERATORS[type(node.op)](left, right)
        if not math.isfinite(float(result)) or abs(result) > 1e100:
            raise ValueError("Result is outside the supported range.")
        return result
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_UNARY_OPERATORS:
        return _ALLOWED_UNARY_OPERATORS[type(node.op)](_safe_eval_node(node.operand))
    raise ValueError("Unsupported expression.")

def calculator_function(expression: str) -> str:
    """Evaluate a basic arithmetic expression; reject all non-arithmetic Python syntax."""
    expression = str(expression).strip()
    if not expression or not _ALLOWED_CHARS.fullmatch(expression):
        return "Error: expression contains unsupported characters."
    try:
        parsed = ast.parse(expression, mode="eval")
        return str(_safe_eval_node(parsed))
    except Exception as exc:
        return f"Error: invalid arithmetic expression ({type(exc).__name__})."

print("182000 * 0.14 ->", calculator_function("182000 * 0.14"))
print("unsafe expression ->", calculator_function("__import__('os').getcwd()"))
print("division by zero ->", calculator_function("1 / 0"))
assert math.isclose(float(calculator_function("182000 * 0.14")), 25480, rel_tol=1e-12)
assert calculator_function("__import__('os').getcwd()").startswith("Error:")

182000 * 0.14 -> 25480.000000000004
unsafe expression -> Error: expression contains unsupported characters.
division by zero -> Error: invalid arithmetic expression (ZeroDivisionError).


Checkpoint: the arithmetic result is 25480.0. The import expression is refused. This AST implementation is intentionally more restrictive than eval() and is the version used by the optional agent below.

## Retrieve internal bank documents
The intended course route is MiniLM embeddings stored in ChromaDB with cosine distance. Those libraries are optional in this execution environment, so the cell below prefers that route when both packages are installed and otherwise uses a TF–IDF retrieval fallback. The fallback is useful for a reproducible local run, but it is not a semantic embedding model.

The retrieval tool returns the two closest documents and preserves document IDs so the agent can cite them.

In [41]:
retrieval_backend = None
retrieval_collection = None
retrieval_model = None

try:
    import chromadb
    from sentence_transformers import SentenceTransformer
    retrieval_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    client = chromadb.Client()
    try:
        client.delete_collection("bank_documents")
    except Exception:
        pass
    retrieval_collection = client.create_collection(
        name="bank_documents",
        metadata={"hnsw:space": "cosine"}
    )
    corpus = (documents["title"] + ". " + documents["text"]).tolist()
    vectors = retrieval_model.encode(corpus, normalize_embeddings=True).tolist()
    retrieval_collection.add(
        ids=documents["id"].tolist(),
        documents=corpus,
        embeddings=vectors,
        metadatas=[{"title": t} for t in documents["title"].tolist()],
    )
    retrieval_backend = "MiniLM embeddings + ChromaDB"
except Exception as exc:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    corpus = (documents["title"] + ". " + documents["text"]).tolist()
    tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
    tfidf_matrix = tfidf.fit_transform(corpus)
    retrieval_backend = "TF-IDF fallback (install sentence-transformers + chromadb for the intended embedding route)"
    print("Embedding/Chroma route unavailable:", type(exc).__name__)

def search_documents_function(query: str, k: int = 2) -> str:
    """Search the bank's fictional internal product, customer, complaints, roadmap, and policy documents. Use for questions about the bank's own data and plans."""
    if retrieval_collection is not None:
        qvec = retrieval_model.encode([query], normalize_embeddings=True).tolist()
        result = retrieval_collection.query(query_embeddings=qvec, n_results=min(k, len(documents)))
        rows = []
        for doc_id, text in zip(result["ids"][0], result["documents"][0]):
            title = documents.loc[documents["id"] == doc_id, "title"].iloc[0]
            rows.append(f"[{doc_id}] {title}: {text}")
        return "\n".join(rows)
    query_vec = tfidf.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix).ravel()
    best = np.argsort(scores)[::-1][:min(k, len(documents))]
    return "\n".join(
        f"[{documents.iloc[i]['id']}] {documents.iloc[i]['title']}: {documents.iloc[i]['text']}"
        for i in best
    )

print("Retrieval backend:", retrieval_backend)
print(search_documents_function("how many customers do we have"))

W1102 19:42:37.797000 4073 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
INFO:sentence_transformers.base.model:No device provided, using mps
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/2_Normalize/config.json "HTTP/1.1 404 Not Found"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieval backend: MiniLM embeddings + ChromaDB


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[customers-1] Customer base: Customer base. In September 2026 the bank had 182,000 active customers, of whom 14% were on the Premium plan. 61% of customers are aged 25 to 44.
[complaints-1] Complaints, third quarter of 2026: Complaints, third quarter of 2026. The bank received 1,240 complaints between July and September 2026: 38% about card delivery, 22% about blocked cards, 15% about transfers and 25% about other topics.


Checkpoint: the customer-base document (customers-1) should be retrieved near the top. Retrieval ranking can vary by backend; always inspect the returned passages rather than assuming the first result is correct.

## Web search, source trimming, and the research agent
Web results are untrusted external context. Keep the title, URL, and a concise content excerpt; preserve the URL so the agent can cite it. A token cap controls prompt size, but aggressive trimming can remove dates, caveats, geography, or the evidence needed to support a claim.

The live agent uses LangChain's create_agent, a Tavily search tool, and the calculator and document tools. If credentials or packages are missing, the notebook still completes its offline checks and clearly marks live execution as skipped.

In [42]:
# Token counting: use tiktoken when installed; fall back to a rough estimate for offline diagnostics.
try:
    import tiktoken
    _encoding = tiktoken.get_encoding("o200k_base")
    def count_tokens(text):
        return len(_encoding.encode(str(text)))
    TOKEN_COUNTER = "tiktoken o200k_base"
except Exception:
    def count_tokens(text):
        return max(1, math.ceil(len(str(text)) / 4))
    TOKEN_COUNTER = "rough character-based estimate (install tiktoken for exact model-family tokenization)"

def trim_to_token_limit(text: str, max_tokens: int = 500) -> str:
    """Trim text to a token budget, retaining a clear truncation marker."""
    if count_tokens(text) <= max_tokens:
        return text
    marker = "\n[TRUNCATED TO FIT CONTEXT BUDGET]"
    budget = max(1, max_tokens - count_tokens(marker))
    if "tiktoken" in TOKEN_COUNTER:
        tokens = _encoding.encode(text)[:budget]
        return _encoding.decode(tokens) + marker
    return text[:budget * 4] + marker

def estimated_cost(input_tokens: int, output_tokens: int) -> float:
    return (input_tokens * INPUT_PRICE_PER_MILLION + output_tokens * OUTPUT_PRICE_PER_MILLION) / 1_000_000

print("Token counter:", TOKEN_COUNTER)
sample_text = "A short offline example of text that can be trimmed to a token budget."
print("Sample token count:", count_tokens(sample_text))

Token counter: tiktoken o200k_base
Sample token count: 15


In [43]:
# Optional live integrations.
research_agent = None
chat_agent = None
live_setup_error = None
try:
    from langchain.agents import create_agent
    from langchain.tools import tool
    from langchain_tavily import TavilySearch
    from langgraph.checkpoint.memory import InMemorySaver
    from langgraph.errors import GraphRecursionError

    if not OPENAI_API_KEY or not TAVILY_API_KEY:
        raise RuntimeError("Set OPENAI_API_KEY and TAVILY_API_KEY in your environment/.env to run live agent exercises.")

    @tool
    def calculator(expression: str) -> str:
        """Safely calculate a basic arithmetic expression. Use for numerical calculations."""
        return calculator_function(expression)

    @tool
    def search_documents(query: str) -> str:
        """Search the bank's fictional internal documents about its plans, customers, complaints, app, fees, and policies. Use for bank-specific facts."""
        return search_documents_function(query)

    tavily_raw = TavilySearch(max_results=3)

    @tool
    def web_search(query: str) -> str:
        """Search the public web for current external facts, competitor information, and recent developments. Cite the returned URLs."""
        raw = tavily_raw.invoke({"query": query})
        # Tavily versions may return a list or a dictionary with a results field.
        results = raw.get("results", []) if isinstance(raw, dict) else raw
        compact = []
        for item in results if isinstance(results, list) else []:
            compact.append({
                "title": item.get("title", ""),
                "url": item.get("url", ""),
                "content": item.get("content", ""),
            })
        return trim_to_token_limit(json.dumps(compact, ensure_ascii=False), 500)

    SYSTEM_PROMPT = (
        "You are a careful research assistant for a fictional bank. "
        "Use search_documents for internal bank facts; calculator for arithmetic; "
        "web_search for current external facts and competitor information. "
        "Do not invent missing facts. Cite every factual claim with an internal document ID "
        "in square brackets or a full source URL. Distinguish internal facts from external facts. "
        "For ambiguous geography or dates, state the limitation. Answer in at most four concise sentences."
    )
    tools = [calculator, search_documents, web_search]
    research_agent = create_agent(model=MODEL, tools=tools, system_prompt=SYSTEM_PROMPT)
    chat_agent = create_agent(
        model=MODEL, tools=tools, system_prompt=SYSTEM_PROMPT,
        checkpointer=InMemorySaver()
    )
    print("Live LangChain agent configured successfully.")
except Exception as exc:
    live_setup_error = f"{type(exc).__name__}: {exc}"
    print("LIVE AGENT NOT CONFIGURED —", live_setup_error)
    print("Offline calculator and document retrieval exercises remain available.")

Live LangChain agent configured successfully.


## Inspect a raw Tavily result and compare its size with the trimmed result
This is a live network call and is intentionally guarded. The bank data is fictional; the example query asks about Revolut's Premium price in Spain. Do not treat a result from another country or an undated comparison page as a reliable answer for Spain in 2026.

In [44]:
if research_agent is not None:
    raw_search = tavily_raw.invoke({"query": "Revolut Premium plan price per month Spain"})
    raw_text = json.dumps(raw_search, ensure_ascii=False)
    compact_search = web_search.invoke({"query": "Revolut Premium plan price per month Spain"})
    print("Raw result tokens:", count_tokens(raw_text))
    print("Trimmed result tokens:", count_tokens(compact_search))
    display(Markdown("**Trimmed web result**\n\n" + compact_search))
else:
    print("Skipped: live Tavily search requires configured API keys and LangChain dependencies.")

Raw result tokens: 1040
Trimmed result tokens: 499


**Trimmed web result**

[{"title": "Personal Fees (Premium) | Revolut Spain", "url": "https://www.revolut.com/en-ES/legal/premium-fees", "content": "## Subscription\n\n### Subscription fee\n\n   €9.99 a month or €99 a year.\n\n## Add money\n\n### Add money\n\n   Free.\n\nHowever, if you add money with a card that has not been issued within the EEA (e.g. a US-based card) or you add money with a commercial card then we may charge a small fee just to cover our costs.\n\n## Card\n\n### First Premium Revolut Card [...] €2.3 per month (this only applies if we decide to close your account and there is still money remaining after the account closure notice period has passed. This fee will be applied to cover our costs for holding and administrating your remaining money. Please see more information about this fee in our Personal terms).\n\n## Revolut Pro Account [...] €2.3 per month (this only applies if we decide to close your account and there is still money remaining after the account closure notice period has passed. This fee will be applied to cover our costs for holding and administrating your remaining money. Please see more information about this fee in our Personal terms).\n\n## Revolut Pro Account"}, {"title": "Compare Revolut plans | Revolut Portugal", "url": "https://www.revolut.com/en-PT/our-pricing-plans", "content": "²The Standard package account is exempt from the payment of a Standard package account Maintenance Fee. Subscription to other package accounts entails a monthly package account Maintenance Fee of €3.99 for the Plus package account, €9.99 for the Premium package account, €17.99 for the Metal package account, and €55 for the Ultra package account, plus Stamp Duty of 4%, which will be charged to the respective current account. [...] ## Premium ### €9.99/month For elevating every day — access exclusive subscriptions, better interest rates, and exchange unlimited amounts of money.\n   ## Metal ### €17.99/month For the global travellers and traders — relax with travel insurance, enjoy enhanced limits, and subscriptions worth €2,600 annually. [...] Standard Plus Premium Metal Ultra\n\nStandard Free²\n\nStandard\n\nPlus€
[TRUNCATED TO FIT CONTEXT BUDGET]

## Ask the research agent three questions
The revenue calculation is based on 182,000 customers × 14% Premium × €9.99/month. The competitor and startup questions require current web evidence and should be checked for country, date, and source quality.

In [45]:
def run_agent(agent, question, config=None):
    """Invoke a LangChain agent and extract answer, tool trace, usage, and estimated cost."""
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config or {"recursion_limit": 12},
    )
    messages = result.get("messages", [])
    ai_messages = [m for m in messages if type(m).__name__ == "AIMessage"]
    input_tokens = sum((m.usage_metadata or {}).get("input_tokens", 0) for m in ai_messages)
    output_tokens = sum((m.usage_metadata or {}).get("output_tokens", 0) for m in ai_messages)
    called_tools = [call.get("name") for m in ai_messages for call in (m.tool_calls or [])]
    last_content = messages[-1].content if messages else ""
    if isinstance(last_content, list):
        last_content = "\n".join(str(x) for x in last_content)
    return {
        "answer": last_content,
        "tools": called_tools,
        "model_calls": len(ai_messages),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "estimated_cost_usd": estimated_cost(input_tokens, output_tokens),
    }

RESEARCH_QUESTIONS = [
    "How much monthly revenue does the Premium plan bring?",
    "How does our Premium plan compare with Revolut's Premium plan in price in Spain?",
    "Name two European fintech startups that raised money for AI products in 2026, with a source for each.",
]
if research_agent is not None:
    research_runs = []
    for question in RESEARCH_QUESTIONS:
        item = run_agent(research_agent, question)
        research_runs.append({"question": question, **item})
        display(Markdown(f"### {question}\n\n{item['answer']}\n\n**Tools:** {item['tools']}  |  **Estimated cost:** ${item['estimated_cost_usd']:.6f}"))
else:
    research_runs = []
    print("Skipped live questions: agent is not configured.")

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


### How much monthly revenue does the Premium plan bring?

The Premium plan costs **€9.99/month**. [premium-1]  
In September 2026, there were **182,000 active customers**, with **14%** on the Premium plan. [customers-1]  
Estimated Premium-plan monthly revenue = 182,000 × 14% × €9.99 ≈ **€254,545.20 per month**. [customers-1] [premium-1]

**Tools:** ['search_documents', 'calculator']  |  **Estimated cost:** $0.000417

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


### How does our Premium plan compare with Revolut's Premium plan in price in Spain?

Our Premium plan in Spain costs **€9.99/month** [premium-1]. Revolut’s Premium plan in Spain costs **€19.99/month** [https://www.revolut.com/en-ES/legal/premium-fees]. So ours is **€10.00/month cheaper**, i.e., about **50% lower** than Revolut’s Premium price in Spain [premium-1] [https://www.revolut.com/en-ES/legal/premium-fees].

**Tools:** ['search_documents', 'web_search', 'search_documents', 'web_search']  |  **Estimated cost:** $0.000656

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


### Name two European fintech startups that raised money for AI products in 2026, with a source for each.

- **Geordie AI (UK)** — raised **$30M (Series A)** to expand its platform for securing and governing enterprise AI agents [https://vestbee.com/insights/articles/top-european-funding-rounds-closed-in-may-2026](https://vestbee.com/insights/articles/top-european-funding-rounds-closed-in-may-2026).  
- **Slamcore (UK)** — raised **$14M** to scale deployment of its visual AI technology across factories and warehouses [https://vestbee.com/insights/articles/top-european-funding-rounds-closed-in-may-2026](https://vestbee.com/insights/articles/top-european-funding-rounds-closed-in-may-2026).

**Tools:** ['web_search', 'web_search', 'web_search']  |  **Estimated cost:** $0.001253

Discussion — why trim search results?
- Cost and latency: every tool result is passed back into the model context, often on multiple agent steps. Fewer tokens generally reduce input-token charges and can shorten latency.
- Attention and grounding: concise results make the most relevant evidence easier to find and reduce distraction from unrelated snippets.
- Trade-off: excessive trimming can remove publication dates, geographic qualifiers, units, methodology, caveats, or source URLs. A good production system should retain source metadata, rank passages, and expand a result when the evidence is insufficient.

## Evaluate tool selection and answer quality
A tool-choice test is useful, but exact set equality can be too strict: an agent may call an additional harmless tool, or calculate mentally when the answer is simple. We report both exact match and required-tool coverage. We also record whether the answer is non-empty. Human review is still required for factual correctness, citations, geography, and temporal accuracy.

In [46]:
TEST_SET = [
    ("What does the Premium plan include?", {"search_documents"}),
    ("How many customers are on the Premium plan?", {"search_documents", "calculator"}),
    ("How many complaints were about card delivery last quarter?", {"search_documents", "calculator"}),
    ("What is 17.5% of 2,400?", {"calculator"}),
    ("What is the monthly price of Revolut's Premium plan in Spain?", {"web_search"}),
    ("What did the European Central Bank decide about interest rates at its latest meeting?", {"web_search"}),
    ("When will our app get an AI assistant, and what must it tell users under the EU AI Act?", {"search_documents", "web_search"}),
    ("Do Premium customers pay a fee on card payments abroad, and how much would a standard customer pay on 300 euros?", {"search_documents", "calculator"}),
]

if research_agent is not None:
    evaluation_rows = []
    for question, expected in TEST_SET:
        outcome = run_agent(research_agent, question)
        called = set(outcome["tools"])
        evaluation_rows.append({
            "question": question,
            "expected_tools": sorted(expected),
            "called_tools": sorted(called),
            "right_tools_exact": called == expected,
            "required_tool_coverage": len(expected & called) / len(expected),
            "answer_nonempty": bool(str(outcome["answer"]).strip()),
            "model_calls": outcome["model_calls"],
            "input_tokens": outcome["input_tokens"],
            "output_tokens": outcome["output_tokens"],
            "estimated_cost_usd": outcome["estimated_cost_usd"],
            "answer": outcome["answer"],
        })
    evaluation_df = pd.DataFrame(evaluation_rows)
    display(evaluation_df.drop(columns=["answer"]))
    print(f"Exact tool-set accuracy: {evaluation_df['right_tools_exact'].mean():.1%}")
    print(f"Mean required-tool coverage: {evaluation_df['required_tool_coverage'].mean():.1%}")
    print(f"Total estimated model cost: ${evaluation_df['estimated_cost_usd'].sum():.6f}")
    display(evaluation_df.loc[~evaluation_df["right_tools_exact"], ["question", "expected_tools", "called_tools", "answer"]])
else:
    evaluation_df = pd.DataFrame()
    print("Skipped: live agent is not configured.")

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


,question,expected_tools,called_tools,right_tools_exact,required_tool_coverage,answer_nonempty,model_calls,input_tokens,output_tokens,estimated_cost_usd
0,What does the Premium plan include?,[search_documents],[search_documents],True,1.0,True,2,729,67,0.000230
1,How many customers are on the Premium plan?,"[calculator, search_documents]","[calculator, search_documents]",True,1.0,True,3,1204,91,0.000355
2,How many complaints were about card delivery last quarter?,"[calculator, search_documents]",[search_documents],False,0.5,True,2,757,130,0.000314
3,"What is 17.5% of 2,400?",[calculator],[calculator],True,1.0,True,2,649,39,0.000179
4,What is the monthly price of Revolut's Premium plan in Spain?,[web_search],[web_search],True,1.0,True,2,1150,59,0.000304
5,What did the European Central Bank decide about interest rates at its latest meeting?,[web_search],[web_search],True,1.0,True,2,1159,99,0.000356
6,"When will our app get an AI assistant, and what must it tell users under the EU AI Act?","[search_documents, web_search]","[search_documents, web_search]",True,1.0,True,2,1336,278,0.000615
7,"Do Premium customers pay a fee on card payments abroad, and how much would a standard customer pay on 300 euros?","[calculator, search_documents]","[calculator, search_documents]",True,1.0,True,3,1240,163,0.000452


Exact tool-set accuracy: 87.5%
Mean required-tool coverage: 93.8%
Total estimated model cost: $0.002802


,question,expected_tools,called_tools,answer
2,How many complaints were about card delivery last quarter?,"[calculator, search_documents]",[search_documents],Last quarter’s complaints about **card delivery** were **38%** of the total complaints for that quarter [complaints-1]. The document doe...


## Interpreting the evaluation
The expected tools in TEST_SET are a rubric, not an absolute law. For example, an LLM might perform 17.5% of 2,400 mentally without invoking the calculator; the answer can be correct even though the trace misses the expected tool. Conversely, calling the expected tool does not prove the answer is correct. A stronger evaluation combines:
- tool selection (required-tool coverage and unnecessary calls),
- answer correctness against a trusted reference,
- citation validity and evidence support,
- latency, token use, and total cost,
- robustness to ambiguous or adversarial questions.
For the card-delivery question, the bank document says 38% of 1,240 complaints were about card delivery: 471.2, approximately 471 complaints. If the question asks for an integer count, state that it is rounded.

## Memory and execution limits
A checkpointer lets an agent retain conversational state across calls sharing the same thread ID. Memory is not a substitute for verifying facts: the assistant should remember the earlier answer but remain able to retrieve the underlying document.

A recursion limit protects the application from unbounded tool/model loops. A user-facing product should explain that the request was too broad, ask for a narrower scope, and log the event for debugging rather than exposing a partial, unverified answer.

In [47]:
if chat_agent is not None:
    thread_config = {"configurable": {"thread_id": "product-team"}, "recursion_limit": 12}
    conversation = [
        "How many complaints did we get last quarter?",
        "What share of them were about blocked cards, and how many is that?",
        "Remind me: what was the total you started from?",
    ]
    for question in conversation:
        answer = run_agent(chat_agent, question, thread_config)
        display(Markdown(f"**User:** {question}\n\n**Assistant:** {answer['answer']}"))
    try:
        from langgraph.errors import GraphRecursionError
        run_agent(
            research_agent,
            "Compare every one of our documents with what our three main competitors offer, one by one.",
            {"recursion_limit": 4},
        )
    except GraphRecursionError:
        print("Stopped by the step limit. User message: This request is too broad to complete safely in one pass. Please choose one product and one competitor.")
else:
    print("Skipped: live agent/checkpointer is not configured.")
    print("Expected memory check: 1,240 complaints; 22% blocked cards (~273); recall total 1,240.")

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


**User:** How many complaints did we get last quarter?

**Assistant:** I don’t have “last quarter” clearly defined in our records, but I can confirm we received **1,240 complaints** in **Q3 2026 (July–September 2026)**. [complaints-1]  
If you tell me which quarter you mean (e.g., **Q2 2026, Q3 2026, or the most recent quarter by date**), I can pull the exact corresponding count.

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


**User:** What share of them were about blocked cards, and how many is that?

**Assistant:** **22%** of the complaints were about **blocked cards**. [complaints-1] That equals **about 273 complaints** (0.22 × 1,240 = 272.8). [complaints-1]

... (4 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Stopped by the step limit. User message: This request is too broad to complete safely in one pass. Please choose one product and one competitor.


## Cost model and scaling thought experiment
The cost helper estimates model token charges from input and output token counts. It does not include Tavily requests, embedding-model inference, vector storage, network transfer, retries, monitoring, or engineering/maintenance time.

Use the actual evaluation results when available. Otherwise, the cells below show the calculation method without inventing a measured API cost.

In [48]:
if not evaluation_df.empty:
    daily_questions = 200
    working_days_per_month = 30
    observed_mean_cost = evaluation_df["estimated_cost_usd"].mean()
    monthly_model_cost = observed_mean_cost * daily_questions * working_days_per_month
    print(f"Observed mean model cost per test question: ${observed_mean_cost:.6f}")
    print(f"Illustrative monthly model-token cost at 200 questions/day: ${monthly_model_cost:.4f}")
    if "tools" in evaluation_df.columns:
        pass
else:
    print("No live token-usage measurements are available in this run.")
    print("Formula: mean_cost_per_question × 200 questions/day × 30 days.")

Observed mean model cost per test question: $0.000350
Illustrative monthly model-token cost at 200 questions/day: $2.1017


Reflection: the cost component to monitor most closely is often repeated input context and web-search volume, not just the final answer. Token accounting also misses the cost of poor answers: staff time spent checking, correcting, and acting on them.

## Extension A — the three additional research prompts
These prompts address the three exercises supplied separately from the ZIP. They are designed to produce structured Markdown, specify a date boundary, require primary/credible sources, and prevent the agent from inventing facts.

The first two tasks are historical research about calendar year 2025. The third compares products whose names and packaging can change; it therefore asks the agent to state the comparison date and distinguish historical Team branding from any successor plan.

In [49]:
prompt_1 = """
Research the most important advancements in generative AI applied to healthcare that were
announced, published, clinically evaluated, or deployed between 1 January and 31 December 2025.
Use web search to retrieve the most recent evidence within that 2025 period. Prioritise peer-reviewed
research, regulators, health systems, and official company or university announcements. Do not
present a press release as proof of clinical effectiveness.

Return clean Markdown with:
1. Executive summary (5 concise bullets).
2. A table with date, organisation/system, healthcare use case, what is new, evidence maturity
   (research / pilot / deployed / regulated), limitations, and source URL.
3. Separate sections for clinical decision support, medical imaging, drug discovery, patient-facing
   assistants, and administrative workflows.
4. Safety, privacy, bias, validation, and regulatory concerns.
5. Five key takeaways and a short source list.

State the search cut-off date. If a claim cannot be verified from a reliable source, label it
'unverified' rather than guessing. Keep the response in Markdown.
"""

prompt_2 = """
Identify notable emerging AI startups that gained meaningful traction, launched important products,
or raised funding during calendar year 2025. Focus on companies founded in or after 2018 and explain
why each is emerging; do not imply that a company is a startup solely because it is popular.

Return clean Markdown with:
1. A short overview of the 2025 AI-startup landscape.
2. A table with company, headquarters, founding year (if verifiable), product, target market,
   technical/product innovation, 2025 milestone or funding round, amount/investors (only if sourced),
   and source URLs.
3. Group companies by sector (e.g. foundation models, agents, healthcare, robotics, developer tools).
4. Explain selection criteria and distinguish funding announcements from product launches.
5. A concise conclusion and source list.

Prioritise company announcements, investor announcements, and reputable business/technology
reporting. Include publication dates. Do not invent funding amounts, investors, or headquarters.
If fewer than five companies meet the evidence standard, return fewer rather than padding the list.
"""

prompt_3 = """
Compare OpenAI ChatGPT Team (including any successor or renamed business offering) with
Anthropic Claude Pro. Make the comparison current as of the date you perform the research, and
clearly distinguish an individual Pro subscription from a multi-seat team/workspace plan.
If discussing the historical ChatGPT Team name, state when it was renamed or superseded.

Return structured Markdown with:
1. A concise verdict explaining that the plans may target different customer types.
2. A comparison table covering intended user, price and billing terms, minimum seats,
   model access, usage limits, context window, file uploads, collaboration/admin controls,
   data use and privacy, integrations, and availability.
3. A separate section on strengths and limitations for individuals, teams, and organisations.
4. A pricing note with currency, region, tax caveats, and the date checked.
5. A recommendation by use case, not a universal winner.
6. Direct source URLs, prioritising official OpenAI and Anthropic pricing, product, and privacy pages.

Do not infer that similarly named plans are equivalent. Mark unavailable or region-dependent
information explicitly, and do not fabricate user reviews or ratings.
"""

for label, prompt in [("Healthcare AI", prompt_1), ("AI startups", prompt_2), ("Product comparison", prompt_3)]:
    print(f"{label}: {len(prompt)} characters; Markdown output and source requirements included.")

Healthcare AI: 1101 characters; Markdown output and source requirements included.
AI startups: 1118 characters; Markdown output and source requirements included.
Product comparison: 1209 characters; Markdown output and source requirements included.


## Extension B — implement a custom use case
Use case: an evidence-grounded product and compliance assistant for a bank. A product manager asks a question such as: When is our in-app AI assistant planned, and what disclosure must it provide? The agent retrieves the internal roadmap and policy, searches authoritative external regulatory sources, and calculates any numerical implications when needed.

The agent should:
- separate internal plans from public regulatory requirements;
- cite each internal document ID and each external URL;
- never send customer names, IBANs, card numbers, or other personal data to external tools;
- state when a regulatory interpretation requires legal review;
- ask for clarification when the jurisdiction or effective date is unclear.
This is a realistic extension of the bank assistant already built above. It adds a privacy boundary and a human-review requirement rather than simply adding more tools.

In [50]:
custom_use_case_prompt = """
You are a product and compliance research assistant for a fictional European bank.
For internal product facts, retrieve the relevant bank document and cite its ID.
For current regulatory requirements, search authoritative sources such as EUR-Lex,
the European Commission, EBA, or national competent authorities and cite direct URLs.
Never invent a legal obligation or imply that a planned product feature is already deployed.
Do not send customer personal data, account identifiers, IBANs, or card numbers to web tools.
Separate (A) internal facts, (B) external regulatory evidence, and (C) recommendations.
Flag legal interpretation for qualified compliance/legal review.
If jurisdiction or date is unclear, ask a clarifying question.
Return concise Markdown with an evidence table.
"""
example_question = "When is Nova planned for launch, and what must the bank's AI assistant disclose to users?"
print("Custom use-case prompt prepared.")
if research_agent is not None:
    print("The base research agent can be queried, but for production use create a dedicated agent with the privacy/compliance system prompt above.")
else:
    print("Live invocation skipped because external agent credentials/dependencies are not configured.")

Custom use-case prompt prepared.
The base research agent can be queried, but for production use create a dedicated agent with the privacy/compliance system prompt above.


## Conclusions and answers to the reflection questions
Why does trimming search results matter?
It reduces repeated context tokens, cost, and distraction. Trimming too much can remove critical dates, geography, caveats, or citations; preserve URLs and expand results when evidence is insufficient.
Was exact tool-set evaluation too strict?
Often, yes. Tool traces measure agent behaviour, not answer correctness. Report exact matches alongside required-tool coverage and independently score answer correctness and citation quality.
What should happen at the step limit?
Show a clear message that the request is too broad, offer a narrower scope, and log the event. Do not present an incomplete answer as verified.
What is missing from token-based cost?
Web-search charges, embedding compute, vector storage, hosting, monitoring, retries, engineering time, and the cost of human review or errors.
Final checklist
- [ ] Calculator accepts basic arithmetic and rejects arbitrary code.
- [ ] Internal retrieval returns relevant document IDs and text.
- [ ] Live web results retain source URLs and are trimmed responsibly.
- [ ] The agent distinguishes internal evidence from external evidence.
- [ ] Evaluation reports both tool-use metrics and answer-quality limitations.
- [ ] Memory is scoped to a thread and step limits are handled gracefully.
- [ ] API keys remain outside the notebook and version control.
- [ ] Historical 2025 research prompts are clearly separated from the original bank-agent pipeline.
